# Reproducing Peron et al. (2020) from CRCNS ssc-11

Paper. Peron S, Pancholi R, Voelcker B, Wittenbach JD, Olafsdottir HF, Freeman J, Svoboda K
(2020). Recurrent interactions in local cortical circuits. *Nature* 579, 256-259.
[doi:10.1038/s41586-020-2062-x](https://doi.org/10.1038/s41586-020-2062-x). The claims file
(`claims/peron-2020.yaml`) was written from the version of record, with its Methods, Extended
Data figure legends, Extended Data Table 1 and reporting summary.

Data set. [CRCNS ssc-11](https://doi.org/10.6080/K0Z31WWG) (doi:10.6080/K0Z31WWG), read
from the public bucket `crcnsarchive` under `ssc-11/live/data/`, one `an<id>.tar.gz` per mouse
(16 files, 120.9 GB compressed). CRCNS data sets are not versioned, so the version is recorded as
`unversioned`; the bucket holds one copy of each file (no `old/` folder), the one described by
the data descriptor version 0.6 (20 February 2020), which is the version the paper's data
statement points to. It is pinned here by the MD5 sums of `checksums.md5`, printed below for
every file read, so that a later change is detectable.

Reading. A gzip stream cannot be read at an offset, so each archive is streamed once from
front to back and every member the claims use is reduced in memory as it passes (helper
`peron2020.py`). Nothing is written to disk. Four archives are read at a time; the full pass
takes 11 to 18 minutes, depending on the connection (648 s in one run and 1,052 s in
another; the time of this run is printed below).

Code. Methods the claims file does not fix are taken from the paper's analysis code,
github.com/peronlab/ablation at commit 74b7321, which the claims file cites; each place is named
in the helper's docstring or the cell that uses it.

| Claim | Tested here |
|---|---|
| model-amplification-increases, model-equal-connectivity-ablation-increases, model-increased-connectivity-ablation-decreases, model-more-ablation-larger-decline, model-slow-input-sensitive, model-response-similarity | Yes: the network model of the Methods, rebuilt and simulated here (last sections) |
| model-all-or-none-robust | No: the Methods' rule for its input strength gives no positive strength here (blocked, see its section) |
| neurons-per-mouse, representation-sizes | Yes |
| touch-ablation-size, whisking-ablation-size, silent-ablation-size | Yes |
| touch-ablation-reduces-touch, touch-ablation-touch-count, touch-ablation-whisking-unchanged | Yes |
| whisking-ablation-no-effect-touch, whisking-ablation-no-effect-whisking | Yes |
| silent-ablation-no-effect-touch, silent-ablation-no-effect-whisking | Yes |
| ablation-type-specificity, ablation-size-correlation | Yes |
| distance-dependence-touch, distance-dependence-whisking-absent | Yes |
| behaviour-unchanged | Yes |
| response-similarity-touch-ablation, response-similarity-example, response-similarity-whisking-ablation | Yes |
| ablation-no-off-target-event-rate, ablation-success-rate | Yes |
| per-mouse-table | Yes |
| glial-reaction-local | No: the histology is not in the data set (blocked, data missing) |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import optimize, stats

import peron2020 as P

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)

started = time.time()
md5 = P.checksums()
data, sizes = P.load_all(workers=4)
print(f"CRCNS {P.DATASET}, version {P.VERSION} (doi:{P.DOI})")
print(f"streamed {len(data)} archives in {time.time() - started:.0f} s")
for a in sorted(data):
    r = data[a]
    print(f"an{a}.tar.gz  md5 {md5['data/an' + a + '.tar.gz']}  {sizes[a] / 1e9:5.2f} GB  "
          f"read {r.get('bytes', 0) / 1e9:5.2f} GB in {r.get('seconds', 0):4.0f} s, reconnections {r.get('reopens')}, "
          f"errors {len(r['errors'])}")

## Data check

What each archive holds, against the data descriptor ("session_neuropilone_????_merged"
directories of `an*_vol_NN_sess_struct.mat` files, and ablation logs) and the paper (16 mice,
24 ablations, 4 to 6 subvolumes of 3 planes, imaging at about 7 Hz).

In [ ]:
rows = []
for a in sorted(data):
    for dname, files in data[a]["sess"].items():
        for fn, r in sorted(files.items()):
            rows.append({"mouse": a, "dir": dname.replace("session_neuropilone_", "").replace("_merged", ""), "file": fn[-19:-16],
                         "format": r["format"], "mouseId": r["mouse"], "whisker": r["whisker"], "cells": len(r["ca_ids"]),
                         "planes": int(r["plane"].max()) + 1, "frames": r["n_frames"], "frame_ms": round(r["ca_dt_ms"], 1),
                         "trials": r["n_trials"], "touches": len(r["touch_on"]), "whisker_dt_ms": r["wh_dt_ms"],
                         "roi_ids_eq_ca_ids": bool(np.array_equal(r["roi_ids"], r["ca_ids"])),
                         "has_trial_avgs": r["hitL"] is not None and r["hitR"] is not None,
                         "trial_avg_frames": None if r["hitL"] is None else r["hitL"].shape[0]})
files = pd.DataFrame(rows)
print("Errors while reading:", sum(len(data[a]["errors"]) for a in data))
print("Session files:", len(files), "; by format:", files.format.value_counts().to_dict())
print("mouseId in file matches archive name:", bool((files.mouseId == "an" + files.mouse).all()))
print("ROI ids of roiArray in the order of caTSA.ids, every file:", bool(files.roi_ids_eq_ca_ids.all()))
print("Files without trial-averaged dF/F features:", files[~files.has_trial_avgs][["mouse", "dir", "file"]].values.tolist())
print("Files with both trial-averaged dF/F features:", int(files.has_trial_avgs.sum()), "of", len(files),
      "; frames in them:", sorted(files.trial_avg_frames.dropna().unique().tolist()))
print("Planes per subvolume file:", files.planes.value_counts().to_dict())
print("Imaging frame interval (ms):", files.frame_ms.min(), "to", files.frame_ms.max(),
      f"({1000 / files.frame_ms.max():.2f} to {1000 / files.frame_ms.min():.2f} Hz)")
print("Whisker video sample interval (ms):", sorted(files.whisker_dt_ms.unique().tolist()))
summ = files.groupby(["mouse", "dir"]).agg(subvolumes=("file", "count"), cells=("cells", "sum"),
                                           trials=("trials", "sum"), whisker=("whisker", "first")).reset_index()
print(summ.to_string(index=False))
print("\nMice:", files.mouse.nunique(), "; whisker tags per mouse:", files.groupby("mouse").whisker.nunique().max(), "(1 = one whisker per mouse)")

In [ ]:
# every member of every archive, by kind
kinds = []
for a in sorted(data):
    for name, size in data[a]["members"]:
        rel = name.split("/", 1)[1] if "/" in name else ""
        k = ("session struct" if rel.endswith("sess_struct.mat") else "JRC ablation log" if "/ablation_log_" in rel
             else "NYU attempt log" if "_attempt_" in rel else "curation file" if (rel.endswith("manualCurationData.mat") or rel.endswith("ablation_ids_manually_curated.mat") or (rel.count("/") == 0 and rel.endswith(".mat")))
             else "directory" if size == 0 else "other")
        kinds.append({"mouse": a, "kind": k, "size": size, "name": rel})
kinds = pd.DataFrame(kinds)
print(kinds.groupby("kind").agg(n=("size", "size"), GB=("size", lambda s: round(s.sum() / 1e9, 1))).to_string())
print("\nOther files:")
oth = kinds[kinds.kind == "other"].copy()
oth["base"] = oth.name.map(lambda n: n.rsplit("/", 1)[-1])
oth["ext"] = oth.base.map(lambda n: n.rsplit(".", 1)[-1].lower() if "." in n else "(none)")
print(f"{len(oth)} files; by extension:", oth.ext.value_counts().to_dict())
for a, g in oth.groupby("mouse"):
    print(f"an{a}: {len(g)} files, {round(g['size'].sum() / 1e6, 1)} MB:", sorted(set(g.base)))
hist = oth[oth.base.str.contains("iba|gfap|histo|slice|section|\\.tif|\\.czi|\\.lsm|\\.nd2|\\.jpg|\\.png", case=False, regex=True)]
print("Files whose name suggests histology or images (iba, gfap, histo, slice, section, tif, czi, lsm, nd2, jpg, png):", len(hist), sorted(set(hist.base))[:20])
print("\nCuration files read:", {a: sorted(data[a]["small"]) for a in sorted(data)})

Duplicates. The analysis code reuses one directory as the post-ablation data of one
ablation and the pre-ablation data of the next (get_data_path.m); that is by design. The check
below looks for the same recording deposited twice: two session files, in any archive, with the
same number of cells, frames and trials and identical touch scores.

In [ ]:
fp = {}
for a in sorted(data):
    for dname, fl in data[a]["sess"].items():
        for fn, r in fl.items():
            t = r["per"]["t"]
            key = (len(r["ca_ids"]), r["n_frames"], r["n_trials"], np.round(np.nan_to_num(t[:50]), 6).tobytes())
            fp.setdefault(key, []).append(f"an{a}/{dname}/{fn}")
dups = [v for v in fp.values() if len(v) > 1]
print("Session files:", sum(len(v) for v in fp.values()), "; distinct recordings:", len(fp), "; duplicated groups:", len(dups))
for v in dups:
    print("  ", v)

Clocks. The encoding scores were computed by the authors; this attempt does not recompute
them from dF/F. It checks that the touch times and the imaging frames share a clock: for each
session file, the mean dF/F of that file's neurons with touch score above 0.1 is aligned to the
first touch of each trial, and to the same times shifted 2 s earlier. A response that rises
after the true touch times and not after the shifted ones shows the alignment.

In [ ]:
rowc = []
for a in sorted(data):
    for dname, fl in data[a]["sess"].items():
        for fn, r in fl.items():
            lags = r["trig_lags"]
            base = lambda v: np.nanmean(v[(lags >= -5) & (lags <= -1)])
            resp = lambda v: np.nanmean(v[(lags >= 1) & (lags <= 6)])
            rowc.append({"mouse": a, "dir": dname, "file": fn, "n_onsets": r["trig_n"],
                         "true": resp(r["trig_real"]) - base(r["trig_real"]),
                         "shifted": resp(r["trig_shift"]) - base(r["trig_shift"])})
clk = pd.DataFrame(rowc)
print(f"Session files: {len(clk)}; response after true touch onsets (mean dF/F, frames 1 to 6 minus -5 to -1): "
      f"median {clk.true.median():.3f}, positive in {(clk.true > 0).sum()}")
print(f"Same for onsets shifted 2 s earlier: median {clk.shifted.median():.3f}, positive in {(clk.shifted > 0).sum()}")
print("Wilcoxon signed-rank, true against shifted, over files: p =", f"{stats.wilcoxon(clk.true, clk.shifted).pvalue:.2g}")
per_mouse = clk.groupby("mouse")[["true", "shifted"]].median().round(3)
print(per_mouse.T.to_string())
print("Files where the true response is not larger than the shifted one:")
print(clk[clk.true <= clk.shifted].to_string(index=False))
r0 = data["272761"]["sess"]["session_neuropilone_preablate_merged"]
r0 = r0[sorted(r0)[0]]
fig, ax = plt.subplots(figsize=(4, 3))
ax.plot(r0["trig_lags"], r0["trig_real"], label="first touch")
ax.plot(r0["trig_lags"], r0["trig_shift"], label="2 s earlier")
ax.set_xlabel("frames from onset"); ax.set_ylabel("mean dF/F, touch-scored cells"); ax.legend(); ax.set_title("an272761 vol 1 pre")
print("Plotted range:", np.round([np.nanmin([r0['trig_real'], r0['trig_shift']]), np.nanmax([r0['trig_real'], r0['trig_shift']])], 3))

### Per-neuron tables

For each of the 24 ablations, the table of `get_ablation_data.m`: pre-ablation and
post-ablation touch and whisking scores, shuffle rejections, event rates, positions, the
ablated neurons of the curation files, and the two proximity exclusions (the Methods cylinder,
the main reading; the code's signed depth rule, a variant). Neurons are matched across periods
by ROI id, as the code does; that a ROI id names the same neuron in the pre and post files is
an assumption the data set does not let us check beyond the ids themselves (the session files
are merged across days by the authors).

In [ ]:
T, INFO = {}, {}
for (a, abl) in P.ABLATIONS:
    T[(a, abl)], INFO[(a, abl)] = P.ablation_table(data[a], a, abl)
info = pd.DataFrame([{"mouse": a, "ablation": abl, **v} for (a, abl), v in INFO.items()])
print(info.to_string(index=False))
d0 = pd.concat([T[k] for k in T])
for c in ["t_pre", "t_post", "w_pre", "w_post", "t_pre_sh", "w_pre_sh", "er_pre", "er_post", "x", "y", "z"]:
    v = d0[c]
    print(f"{c:9s} range {np.nanmin(v):8.3f} to {np.nanmax(v):8.3f}  median {np.nanmedian(v):7.3f}  NaN {int(v.isna().sum())}")
print("Units: scores are correlations (unitless), event rates Hz, positions um (600 um / 512 pixels; planes 15 um apart; subvolumes 45 um apart, as the code).")
print("Every ablated id found among the neurons:", bool((info.n_ablated_ids == info.n_ablated_found).all()))
print("Neurons dropped as not in both periods:", info.n_dropped_unmatched.sum(), "(all in", info[info.n_dropped_unmatched > 0][["mouse", "ablation"]].values.tolist(), ")")

In [ ]:
# neurons with a NaN event rate fall in no event-rate decile, so the shuffle test never rejects them
rowsn = []
for (a, abl), d in T.items():
    for per in ["pre", "post"]:
        nn = d[f"er_{per}"].isna()
        rowsn.append({"mouse": a, "ablation": abl, "period": per, "nan_rate": int(nn.sum()),
                      "of_which_t_above_0.1": int((nn & (d[f"t_{per}"] > 0.1)).sum()), "of_which_w_above_0.1": int((nn & (d[f"w_{per}"] > 0.1)).sum())})
rowsn = pd.DataFrame(rowsn)
print(rowsn[rowsn.nan_rate > 0].to_string(index=False))
print("Neurons with a NaN event rate, over all tables and periods:", int(rowsn.nan_rate.sum()),
      "; with a touch or whisking score above 0.1:", int(rowsn["of_which_t_above_0.1"].sum() + rowsn["of_which_w_above_0.1"].sum()))
memb_nan = sum(int((P.members(d, r) & (d.er_pre.isna() | d.er_post.isna())).sum()) for d in T.values() for r in ["t", "w"])
print("Members (either representation, any ablation) with a NaN event rate in either period:", memb_nan)

## neurons-per-mouse

> Volumetric imaging in L2/3 of the spared whisker's barrel column recorded 8,126 ± 2,436
> (mean ± s.d.) neurons per mouse across 16 mice.

The first ablation of each mouse, as ablation_summ_stats.m (`first_anims`). The count is the
size of the per-neuron table, which keeps only neurons present in both periods; the raw count of
the pre-ablation files is shown too.

In [ ]:
FIRST = {"touch": ["250220", "257220", "274424", "272761", "274577"],
         "silent": ["257218", "258836", "271211", "278937", "278939", "275801", "275798", "276013"],
         "whisking": ["281915", "278288", "278759"]}
# Extended Data Table 1, transcribed from the PDF of the paper (reference values, not results)
EDT1 = pd.DataFrame([
    ("250220", "touch", 8844, 50, 867, -0.122, 422, -0.076), ("257218", "silent", 6215, 20, 743, 0.018, 366, 0.032),
    ("257220", "touch", 3663, 9, 389, -0.004, 455, 0.025), ("258836", "silent", 7290, 16, 481, 0.066, 666, -0.012),
    ("258836", "touch", None, 15, 488, -0.055, 655, 0.001), ("271211", "silent", 9415, 16, 2476, -0.012, 551, -0.007),
    ("271211", "touch", None, 15, 1951, -0.061, 692, 0.021), ("278937", "silent", 4230, 15, 146, 0.004, 379, 0.001),
    ("278937", "whisking", None, 12, 150, -0.006, 392, 0.011), ("278939", "silent", 7167, 20, 636, -0.024, 698, 0.001),
    ("278939", "whisking", None, 13, 523, -0.017, 649, -0.015), ("281915", "whisking", 7002, 8, 723, -0.010, 550, 0.004),
    ("281915", "touch", None, 11, 601, -0.022, 672, 0.008), ("274424", "touch", 7785, 17, 706, -0.029, 718, -0.014),
    ("272761", "touch", 6359, 14, 509, -0.001, 410, 0.000), ("274577", "touch", 12173, 11, 1380, -0.013, 657, 0.004),
    ("275801", "silent", 8354, 14, 220, 0.000, 738, -0.007), ("275801", "touch", None, 9, 197, -0.065, 698, -0.006),
    ("275798", "silent", 11364, 13, 959, 0.002, 958, 0.002), ("275798", "whisking", None, 11, 972, 0.008, 1037, 0.011),
    ("278288", "whisking", 9314, 25, 969, -0.008, 813, -0.006), ("276013", "silent", 11659, 16, 652, 0.028, 1072, -0.001),
    ("276013", "whisking", None, 9, 634, -0.012, 968, -0.008), ("278759", "whisking", 9187, 11, 617, -0.003, 526, -0.004)],
    columns=["mouse", "ablation", "cells", "ablated", "touch", "dRtouch", "whisking", "dRwhisking"])
first = []
for abl, mice in FIRST.items():
    for a in mice:
        d = T[(a, abl)]
        first.append({"mouse": a, "first_ablation": abl, "cells_table": len(d), "cells_raw_pre": INFO[(a, abl)]["n_pre_raw"],
                      "EDT1_cells": int(EDT1[(EDT1.mouse == a) & (EDT1.ablation == abl)].cells.iloc[0]),
                      "touch_a": P.class_counts(d, "t", "a")[0], "whisking_a": P.class_counts(d, "w", "a")[0],
                      "touch_b": P.class_counts(d, "t", "b")[0], "whisking_b": P.class_counts(d, "w", "b")[0]})
first = pd.DataFrame(first)
print(first.to_string(index=False))
print(f"\nNeurons per mouse (table size): mean {first.cells_table.mean():.1f}, s.d. {first.cells_table.std():.1f} (paper 8,126 +/- 2,436; n = {len(first)})")
print(f"Raw pre-ablation count (variant): mean {first.cells_raw_pre.mean():.1f}, s.d. {first.cells_raw_pre.std():.1f}")
print("Per-mouse counts equal to Extended Data Table 1:", int((first.cells_table == first.EDT1_cells).sum()), "of", len(first))

**Result: reproduced.** The 16 per-neuron tables hold 8,126.3 ± 2,435.6 neurons per mouse,
and every per-mouse count equals the Cell count column of Extended Data Table 1. In an278937 the
pre-ablation files hold 995 more neurons (a sixth subvolume) than the post-ablation files, and
the table, like the paper, counts only the neurons in both.

## representation-sizes

> Across 16 mice, 901 ± 539 of the imaged neurons encoded touch (fraction 0.108 ± 0.051) and
> 865 ± 364 encoded whisking (fraction 0.106 ± 0.028).

Readings of the criterion: (a) score above 0.1 alone in the first pre-ablation data
(ablation_summ_stats.m); (b) the Methods membership rule in that period.

In [ ]:
for lab, tc, wc in [("(a) score > 0.1", "touch_a", "whisking_a"), ("(b) Methods membership", "touch_b", "whisking_b")]:
    ft, fw = first[tc] / first.cells_table, first[wc] / first.cells_table
    print(f"{lab:24s} touch {first[tc].mean():6.1f} +/- {first[tc].std():5.1f} (fraction {ft.mean():.3f} +/- {ft.std():.3f});  "
          f"whisking {first[wc].mean():6.1f} +/- {first[wc].std():5.1f} (fraction {fw.mean():.3f} +/- {fw.std():.3f})")
print("Paper: touch 901 +/- 539 (0.108 +/- 0.051), whisking 865 +/- 364 (0.106 +/- 0.028)")

**Result: reproduced.** Under reading (a) the mean counts are 900.6 touch and 864.6 whisking
neurons, fractions 0.108 and 0.106, all four within tolerance (and the spreads, 538.8 and
364.4, as printed in the paper). Reading (b) gives 578.1 and 453.8, outside tolerance, so the
paper's text counts by score alone.

## touch-ablation-size, whisking-ablation-size, silent-ablation-size

> Touch ablations removed 16.8 ± 12.8 strong touch neurons per mouse (about 6% of the touch
> neurons of the barrel column, 9 mice), whose touch scores were at the 70th ± 35th percentile.
> Whisking ablations removed 12.7 ± 5.7 strong whisking neurons per mouse (about 4%, 7 mice),
> at the 66th ± 37th percentile of whisking score. Silent-neuron ablations (event rate below
> 0.025 Hz) removed 16.3 ± 2.6 neurons per mouse in 8 mice.

Two percentile rules, pooled over ablated neurons. The criterion's rule is a rank: the fraction
of included neurons whose score is at or below the ablated neuron's (1 when none exceeds it;
ablated neurons with no score are left out). The code's rule (ablation_summ_stats.m) is the
position of the first included neuron whose score exceeds the ablated neuron's, over the number
of included neurons; when none exceeds it, or the ablated neuron has no score, the position is
the first, so the value is 1/N (the synthetic check below gives 0.01 for a score above all
100).

In [ ]:
def pct(d, rank_r, own_r):
    """The code's rule (ablation_summ_stats.m): position of the first included neuron whose
    score exceeds the ablated neuron's, over the number included. When none exceeds it, or
    the ablated neuron's score is NaN, the position is the first one, so the value is 1/N."""
    ss = np.sort(d[f"{rank_r}_pre"][P.members(d, rank_r)].values)
    q = []
    for v in d[d.is_ablated == 1][f"{own_r}_pre"].values:
        ii = np.where(ss > v)[0]
        q.append((ii[0] + 1 if len(ii) else 1) / len(ss))
    return q

def pct_rank(d, rank_r, own_r):
    """The criterion's rule: rank of the ablated neuron's score among the included neurons'
    scores, as the fraction of them at or below it (1 when none exceeds it); NaN scores have
    no rank and are left out."""
    ss = np.sort(d[f"{rank_r}_pre"][P.members(d, rank_r)].values)
    v = d[d.is_ablated == 1][f"{own_r}_pre"].values
    v = v[~np.isnan(v)]
    return list(np.searchsorted(ss, v, side="right") / len(ss))

def above_all(d, rank_r, own_r):
    ss = d[f"{rank_r}_pre"][P.members(d, rank_r)].values
    v = d[d.is_ablated == 1][f"{own_r}_pre"].values
    return int((v > np.nanmax(ss)).sum()), int(np.isnan(v).sum()), len(v)

size = {}
for abl in ["touch", "whisking", "silent"]:
    n = np.array([T[(a, abl)].is_ablated.sum() for a in P.ANIMS[abl]])
    size[abl] = n
    print(f"{abl:8s} ablated per mouse {n.tolist()}: mean {n.mean():.2f}, s.d. {n.std(ddof=1):.2f}, mean/287 = {100 * n.mean() / 287:.2f}%")
# a synthetic check of the two rules: 100 included scores 1..100, ablated scores 50.5 and 101
_d = pd.DataFrame({"t_pre": np.r_[np.arange(1, 101), 50.5, 101.0], "is_ablated": np.r_[np.zeros(100), 1, 1]})
_m = P.members
P.members = lambda d, r, thresh=0.1: d.is_ablated == 0
print("synthetic: code rule", pct(_d, "t", "t"), "rank rule", pct_rank(_d, "t", "t"))
P.members = _m
for lab, abl, rr, ro in [("touch", "touch", "t", "t"), ("whisking, Methods (Rwhisking among whisking neurons)", "whisking", "w", "w"),
                         ("whisking, code (Rwhisking among touch neurons' Rtouch)", "whisking", "t", "w")]:
    q_r = sum([pct_rank(T[(a, abl)], rr, ro) for a in P.ANIMS[abl]], [])
    q_c = sum([pct(T[(a, abl)], rr, ro) for a in P.ANIMS[abl]], [])
    aa = np.array([above_all(T[(a, abl)], rr, ro) for a in P.ANIMS[abl]]).sum(0)
    print(f"{lab} percentile: rank rule (criterion) mean {100 * np.mean(q_r):.1f}, s.d. {100 * np.std(q_r, ddof=1):.1f} (n = {len(q_r)}); "
          f"code rule mean {100 * np.mean(q_c):.1f}, s.d. {100 * np.std(q_c, ddof=1):.1f} (n = {len(q_c)}); "
          f"ablated neurons scoring above every included neuron {aa[0]}, with a NaN score {aa[1]}, of {aa[2]}")
for abl, mice in P.ANIMS.items():
    print(abl, "per mouse percentile (rank rule):", {a: round(100 * np.mean(pct_rank(T[(a, abl)], 't' if abl == 'touch' else 'w', 't' if abl == 'touch' else 'w')), 1) for a in mice} if abl != "silent" else "")
er = np.concatenate([T[(a, "silent")].er_pre[T[(a, "silent")].is_ablated == 1].values for a in P.ANIMS["silent"]])
print(f"silent: pre-ablation event rate of the {len(er)} ablated neurons: median {np.median(er):.4f} Hz, max {er.max():.4f} Hz; "
      f"{(er < 0.025).sum()} below 0.025 Hz")
print("Ablated counts per ablation equal to Extended Data Table 1:",
      int(sum(T[(r.mouse, r.ablation)].is_ablated.sum() == r.ablated for r in EDT1.itertuples())), "of 24")

**Result: touch-ablation-size partially reproduced.** 16.78 ± 12.77 neurons per mouse and 5.85%
of 287 are within tolerance. The percentile under the criterion's rule (the rank of each ablated
neuron's score among the included touch neurons) is 79.3 ± 27.1 over the 145 ablated neurons
with a score, outside 70 ± 7 and 35 ± 3.5. The code's rule gives 70.3 ± 35.0, the paper's
values: it puts the 9 ablated neurons that score above every included neuron, and the 6 with no
score, at 1/N in place of the top (the synthetic check gives 0.01 for a score above all 100).
The paper's percentile is therefore that rule's, and the gap is asked of a reader.

**Result: whisking-ablation-size partially reproduced.** 12.71 ± 5.68 neurons are within
tolerance; 4.43% of 287 is outside 4 ± 0.4. Under the criterion's rank rule the percentile is
75.4 ± 31.8 (Rwhisking among whisking neurons) or 75.8 ± 30.2 (among the touch neurons' Rtouch),
both outside 66 ± 6.6; the code's rule gives 68.8 ± 36.2 and 65.9 ± 36.5, within tolerance.

**Result: silent-ablation-size reproduced.** 16.25 ± 2.55 neurons per mouse. Of the 130
ablated neurons, 118 have a pre-ablation event rate below 0.025 Hz in these data (the highest
is 0.079 Hz). The paper gives the 0.025 Hz rule for choosing targets; 12 of the 130 ablated
neurons are above it in these pre-ablation data.

## Score changes by ablation type

The common method of the claims file: per mouse, the median pre and post score of the members of
a representation; grand median across mice; Wilcoxon signed-rank test on paired mouse medians.
Counts of members per period follow touch-ablation-touch-count. The Methods cylinder is the main
reading and the code's rule a variant.

In [ ]:
def type_summary(abl, r, cyl="methods"):
    rows = []
    for a in P.ANIMS[abl]:
        d = P.with_cyl(T[(a, abl)], cyl)
        pre, post, n = P.mouse_medians(d, r)
        ca = P.class_counts(d, r, "a")
        cad = P.class_counts(d, r, "a", keep_ablated=False)
        cb = P.class_counts(d, r, "b")
        rows.append({"mouse": a, "pre": pre, "post": post, "change": post - pre, "n": n, "n_a_pre": ca[0], "n_a_post": ca[1],
                     "n_adrop_pre": cad[0], "n_adrop_post": cad[1], "n_b_pre": cb[0], "n_b_post": cb[1],
                     "score_ablated": float(np.nansum(d.t_pre[d.is_ablated == 1]))})
    return pd.DataFrame(rows)

def show(abl, r):
    out = {}
    for cyl in ["methods", "code"]:
        s = type_summary(abl, r, cyl)
        p = stats.wilcoxon(s.pre, s.post).pvalue
        out[cyl] = (s, p)
        print(f"[{cyl} cylinder] {abl} ablation, {'Rtouch' if r == 't' else 'Rwhisking'}: grand median {s.pre.median():.3f} -> {s.post.median():.3f} "
              f"(adjusted MAD {P.adj_mad(s.pre):.3f}, {P.adj_mad(s.post):.3f}); neurons {s.n.sum()}; signed-rank p = {p:.3f}; "
              f"median change {s.change.median():+.4f}, mice with a decline {(s.change < 0).sum()} of {len(s)}")
    s = out["methods"][0]
    print(s.round(4).to_string(index=False))
    print(f"Counts, mean over mice: (a) score > 0.1, ablated kept {s.n_a_pre.mean():.1f} -> {s.n_a_post.mean():.1f}; "
          f"(a) ablated left out {s.n_adrop_pre.mean():.1f} -> {s.n_adrop_post.mean():.1f}; (b) Methods rule {s.n_b_pre.mean():.1f} -> {s.n_b_post.mean():.1f}")
    print(f"Sum over mice of the (a) pre-ablation count: {s.n_a_pre.sum()}")
    return out

R = {}

### touch-ablation-reduces-touch, touch-ablation-touch-count

> Ablating touch neurons reduced the touch-encoding score of the spared touch neurons, from
> 0.123 to 0.100 (grand median across 9 mice, 8,392 neurons). After touch ablation the number
> of touch neurons fell, from 932 ± 634 to 716 ± 469 (mean ± s.d. over 9 mice), ablated
> neurons excluded.

In [ ]:
R[("touch", "t")] = show("touch", "t")

**Result: touch-ablation-reduces-touch reproduced.** Grand median Rtouch 0.123 -> 0.100,
signed-rank p = 0.004 (all 9 mice decline), within tolerance under both cylinders. The
criterion's members number 7,121 neurons (7,089 with the code's cylinder), against the paper's
8,392; 8,392 is the sum over the 9 mice of the pre-ablation count of neurons with score above
0.1 (reading (a) of the counts), which is 8,392 here, so the paper's neuron count
is that sum, not the number of neurons entering the medians.

**Result: touch-ablation-touch-count reproduced.** Under reading (a) the mean count goes from
932.4 to 715.8 with ablated neurons kept (the code), and 918.3 to 712.6 with them left out
(the text); both within tolerance. Reading (b) gives 623.7 -> 472.6.

### touch-ablation-whisking-unchanged

> Touch ablation did not change the whisking-encoding score of whisking neurons (0.116 to
> 0.115, 6,975 neurons, P = 0.820); the whisking neuron count went from 775 ± 267 to 721 ± 241.

In [ ]:
R[("touch", "w")] = show("touch", "w")

**Result: reproduced.** Rwhisking 0.116 -> 0.115, signed-rank p = 0.820 (no change at the 0.05
level); mean counts 775.0 -> 721.2 (reading (a)); 5,389 member neurons (the sum of the (a)
pre counts is 6,975, the paper's neuron count).

### whisking-ablation-no-effect-touch, whisking-ablation-no-effect-whisking

> Ablating whisking neurons had no effect on the touch representation (Rtouch 0.112 to 0.095,
> 7 mice, 5,866 neurons, P = 0.109; count 838 ± 401 to 899 ± 548). Ablating whisking neurons did
> not degrade the whisking representation (Rwhisking 0.108 to 0.107, 6,161 neurons, P = 0.812;
> count 880 ± 387 to 974 ± 558).

In [ ]:
R[("whisking", "t")] = show("whisking", "t")

In [ ]:
R[("whisking", "w")] = show("whisking", "w")

### silent-ablation-no-effect-touch, silent-ablation-no-effect-whisking

> Ablating silent neurons did not change the touch representation (Rtouch 0.115 to 0.107, 8
> mice, 7,110 neurons, P = 0.383; count 889 ± 713 to 926 ± 681). Ablating silent neurons did not
> change the whisking representation (Rwhisking 0.115 to 0.114, 7,684 neurons, P = 0.844; count
> 961 ± 464 to 904 ± 384).

In [ ]:
R[("silent", "t")] = show("silent", "t")

In [ ]:
R[("silent", "w")] = show("silent", "w")

Where the paper's grand medians for whisking and silent ablations come from:
ablation_summ_stats.m fills one matrix of mouse medians, indexed by the ablation's position in
the list of all 24 ablations (touch 1 to 9, silent 10 to 17, whisking 18 to 24), and takes the
median over the whole matrix. It prints the whisking block before the silent rows are filled
(they are zeros then), and the silent block when all 24 rows are filled. The cell below
recomputes both, from this attempt's mouse medians.

In [ ]:
order = [("touch", a) for a in P.ANIMS["touch"]] + [("silent", a) for a in P.ANIMS["silent"]] + [("whisking", a) for a in P.ANIMS["whisking"]]
for r, lab in [("t", "Rtouch"), ("w", "Rwhisking")]:
    M = np.array([P.mouse_medians(T[(a, abl)], r)[:2] for abl, a in order])
    Mz = M.copy(); Mz[9:17] = 0
    print(f"{lab}: median over all 24 ablations {np.median(M[:, 0]):.3f} -> {np.median(M[:, 1]):.3f}   "
          f"median with the 8 silent rows zero (whisking block) {np.median(Mz[:, 0]):.3f} -> {np.median(Mz[:, 1]):.3f}")
print("Paper: silent Rtouch 0.115 -> 0.107, Rwhisking 0.115 -> 0.114; whisking Rtouch 0.112 -> 0.095, Rwhisking 0.108 -> 0.107")

In [ ]:
# the paper's +/- values against two definitions of the adjusted MAD: 1.4826 x median absolute
# deviation (used above), and 1.4826 x MATLAB's default mad (mean absolute deviation)
mean_mad = lambda x: 1.4826 * np.mean(np.abs(np.asarray(x) - np.mean(x)))
paper_pm = {("touch", "t"): (0.021, 0.037), ("touch", "w"): (0.013, 0.024), ("whisking", "t"): (0.081, 0.069),
            ("whisking", "w"): (0.076, 0.076), ("silent", "t"): (0.021, 0.023), ("silent", "w"): (0.014, 0.015)}
for (abl, r), (pp, pq) in paper_pm.items():
    s = R[(abl, r)]["methods"][0]
    print(f"{abl:8s} {'Rtouch' if r == 't' else 'Rwhisking':9s}: median-based {P.adj_mad(s.pre):.3f} / {P.adj_mad(s.post):.3f}; "
          f"mean-based {mean_mad(s.pre):.3f} / {mean_mad(s.post):.3f}; s.d. {s.pre.std(ddof=1):.3f} / {s.post.std(ddof=1):.3f}; paper {pp} / {pq}")
# the whisking and silent medians of the paper come from the 24-row matrix (cell above); the same matrix for the spread
for r in ["t", "w"]:
    M = np.array([P.mouse_medians(T[(a, abl)], r)[:2] for abl, a in order])
    Mz = M.copy(); Mz[9:17] = 0
    for lab, X in [("whisking block, 8 zero rows", Mz), ("all 24 rows", M)]:
        print(f"{'Rtouch' if r == 't' else 'Rwhisking'} {lab}: median-based {P.adj_mad(X[:, 0]):.3f} / {P.adj_mad(X[:, 1]):.3f}; "
              f"mean-based {mean_mad(X[:, 0]):.3f} / {mean_mad(X[:, 1]):.3f}")

The paper's ± values are reproduced by 1.4826 times the mean absolute deviation (MATLAB's `mad`
with its default flag), not by the median absolute deviation the text names: touch ablation
Rtouch 0.021 / 0.038 against 0.021 / 0.037, and Rwhisking 0.013 / 0.024, equal to the paper. For
the whisking and silent ablations they match only on the same matrices as the paper's medians
(whisking block with eight zero rows 0.081 / 0.070 and 0.076 / 0.076; all 24 rows 0.021 / 0.023
and 0.014 / 0.015). The ± values above (median-based) are therefore about half the paper's for the touch
ablations (0.011 / 0.016 against 0.021 / 0.037), and about a tenth for the whisking ablations
(Rtouch 0.008 / 0.009 against 0.081 / 0.069), whose paper values come from the zero-padded
matrix.

**Result: whisking-ablation-no-effect-touch reproduced.** Signed-rank p = 0.109 (Rtouch 0.113 ->
0.106 over the 7 whisking ablation mice; 4,605 member neurons), no change at the 0.05 level. The
paper's 0.112 -> 0.095 are reproduced by the code's median over a matrix with eight zero rows
(0.112 -> 0.096), not by the median over the 7 mice; the post value 0.106 is outside the
claim's tolerance (0.085 to 0.105), but the criterion of this claim of absence is the test.
Counts 838.0 -> 898.9 (reading (a)), and the (a) pre counts sum to 5,866, the paper's neuron count.

**Result: whisking-ablation-no-effect-whisking reproduced.** Signed-rank p = 0.938 with the
Methods cylinder and 0.812 with the code's (the paper's value). Rwhisking 0.114 -> 0.110 over the
7 mice; the paper's 0.108 -> 0.107 is again the zero-padded median (0.109 -> 0.107 here).
Counts 880.1 -> 973.7; the (a) pre counts sum to 6,161.

**Result: silent-ablation-no-effect-touch reproduced.** Signed-rank p = 0.383; Rtouch 0.111 ->
0.113 over the 8 mice. The paper's 0.115 -> 0.107 equal the median over all 24 ablations.
Counts 888.8 -> 925.8; the (a) pre counts sum to 7,110.

**Result: silent-ablation-no-effect-whisking reproduced.** Signed-rank p = 0.844; Rwhisking
0.114 -> 0.115 over the 8 mice (the paper's 0.115 -> 0.114 are close to the 24-ablation median,
0.115 -> 0.113). Counts 960.5 -> 904.4; the (a) pre counts sum to 7,684.

## ablation-type-specificity

> Touch ablations produced significantly different changes in Rtouch from whisking or silent
> ablations, while whisking and silent ablations did not differ; for Rwhisking no pair of
> ablation types differed.

Wilcoxon rank-sum test on the per-mouse changes. With 9, 8 and 7 mice MATLAB's `ranksum` uses
its exact method; scipy's `mannwhitneyu` with `method="exact"` is the same test.

In [ ]:
for cyl in ["methods", "code"]:
    for r, lab in [("t", "Rtouch"), ("w", "Rwhisking")]:
        ch = {abl: R[(abl, r)][cyl][0].change.values for abl in ["touch", "silent", "whisking"]}
        p = lambda x, y: stats.mannwhitneyu(ch[x], ch[y], method="exact").pvalue
        print(f"[{cyl}] change in {lab}: touch-silent p = {p('touch', 'silent'):.3f}, touch-whisking p = {p('touch', 'whisking'):.3f}, "
              f"whisking-silent p = {p('whisking', 'silent'):.3f};  median changes touch {np.median(ch['touch']):+.4f}, "
              f"silent {np.median(ch['silent']):+.4f}, whisking {np.median(ch['whisking']):+.4f}")
print("Paper: Rtouch 0.002, 0.042, 0.152; Rwhisking 0.743, 0.758, 1")
# where 0.031 and the paper's 0.042 part: the same test on the table's own per-mouse changes
ch_t = {abl: R[(abl, "t")]["code"][0].set_index("mouse").change for abl in ["touch", "whisking"]}
for abl in ["touch", "whisking"]:
    e = EDT1[EDT1.ablation == abl].set_index("mouse").dRtouch
    print(f"{abl}: change in Rtouch here (code cylinder) and in Extended Data Table 1, by mouse:",
          {a: (round(ch_t[abl][a], 3), e[a]) for a in e.index})
et, ew = EDT1[EDT1.ablation == "touch"].dRtouch.values, EDT1[EDT1.ablation == "whisking"].dRtouch.values
print(f"rank-sum on the table's values, touch against whisking: p = {stats.mannwhitneyu(et, ew, method='exact').pvalue:.3f}; "
      f"with ties (asymptotic, as MATLAB ranksum uses for ties): p = {stats.mannwhitneyu(et, ew, method='asymptotic').pvalue:.3f}")
allv = np.r_[ch_t["touch"].values, ch_t["whisking"].values]
rk = stats.rankdata(allv)
print("ranks here, touch:", rk[:9].astype(int).tolist(), "whisking:", rk[9:].astype(int).tolist())
rk_e = stats.rankdata(np.r_[et, ew])
print("ranks in the table, touch:", rk_e[:9].tolist(), "whisking:", rk_e[9:].tolist())

**Result: reproduced.** For Rtouch, touch-silent p = 0.002 and touch-whisking p = 0.031, while
whisking-silent p = 0.152; for Rwhisking all three are above 0.05 (0.743, 0.837, 0.955 with the
Methods cylinder; 0.743, 0.758, 1.000 with the code's, the paper's values). Only the
touch-whisking value differs from the paper's (0.031 against 0.042). The same test on the
table's own per-mouse changes gives 0.042: an278937's whisking-ablation change is -0.003 here and
-0.006 in the table, which swaps its rank (13 here, 12 in the table) with an257220's touch
ablation (12 here, 13 in the table).

## ablation-size-correlation

> The decline in the touch representation grew with the touch representation ablated, the
> change in Rtouch correlating with the net Rtouch ablated (R = -0.794 over all 24 ablations;
> R = -0.779 over the 9 touch ablations).

In [ ]:
for cyl in ["methods", "code"]:
    allc = pd.concat([R[(abl, "t")][cyl][0].assign(ablation=abl) for abl in ["touch", "silent", "whisking"]])
    r_all = stats.pearsonr(allc.score_ablated, allc.change)
    tt = allc[allc.ablation == "touch"]
    r_t = stats.pearsonr(tt.score_ablated, tt.change)
    print(f"[{cyl}] all 24: R = {r_all.statistic:.3f}, p = {r_all.pvalue:.2g};  touch 9: R = {r_t.statistic:.3f}, p = {r_t.pvalue:.3f}")
    if cyl == "methods":
        print(allc[["ablation", "mouse", "score_ablated", "change"]].round(3).to_string(index=False))
        fig, ax = plt.subplots(figsize=(4, 3))
        for abl, c in [("touch", "C0"), ("silent", "k"), ("whisking", "C2")]:
            s = allc[allc.ablation == abl]
            ax.scatter(s.score_ablated, s.change, c=c, label=abl)
        ax.set_xlabel("net Rtouch ablated"); ax.set_ylabel("change in median Rtouch"); ax.legend()
        print("Plotted ranges: x", np.round([allc.score_ablated.min(), allc.score_ablated.max()], 2), "y", np.round([allc.change.min(), allc.change.max()], 3))

**Result: reproduced.** R = -0.790 over the 24 ablations (p = 4.4e-06) and R = -0.761 over the 9
touch ablations (p = 0.017), both within tolerance and significant (-0.797 and -0.781 with the
code's cylinder). One mouse (an250220, 50
neurons ablated) carries much of the touch-ablation correlation.

## distance-dependence-touch, distance-dependence-whisking-absent

> After touch ablation, touch neurons 15-35 um from the nearest ablated neuron lost more Rtouch
> than those 115-135 um away (-0.132 against -0.056), and the effect decayed with distance with
> a length constant of 87 um. Touch ablation produced no distance-dependent change in the
> whisking representation (proximal 0.000, distal -0.023).

Readings: (a) the code's (per-mouse means for the paired test; median over all neurons pooled
for the values); (b) per-mouse medians, grand median as the value. Length constant: fit of
a exp(b d) to the cross-mouse mean of per-mouse mean changes in 25-um bins from 0 to 150 um.

In [ ]:
def distance(r, cyl):
    pm, dm, pmed, dmed, allp, alld, byb = [], [], [], [], [], [], []
    bins = np.arange(0, 151, 25)
    for a in P.ANIMS["touch"]:
        d = P.with_cyl(T[(a, "touch")], cyl)
        m = P.members(d, r)
        dd = d.d_ablated_um[m].values
        ds = (d[f"{r}_post"] - d[f"{r}_pre"])[m].values
        px, di = (dd > 15) & (dd < 35), (dd > 115) & (dd < 135)
        pm.append(np.nanmean(ds[px])); dm.append(np.nanmean(ds[di]))
        pmed.append(np.nanmedian(ds[px])); dmed.append(np.nanmedian(ds[di]))
        allp += list(ds[px]); alld += list(ds[di])
        byb.append([np.nanmean(ds[(dd >= bins[i]) & (dd < bins[i + 1])]) for i in range(len(bins) - 1)])
    x = bins[:-1] + 12.5
    y = np.nanmean(np.array(byb), 0)
    popt, _ = optimize.curve_fit(lambda x, a, b: a * np.exp(b * x), x, y, p0=(y[0], -0.01), maxfev=20000)
    return dict(pm=pm, dm=dm, pmed=pmed, dmed=dmed, allp=allp, alld=alld, x=x, y=y, lam=abs(1 / popt[1]), byb=np.array(byb))

DIST = {}
for r in ["t", "w"]:
    for cyl in ["methods", "code"]:
        o = distance(r, cyl); DIST[(r, cyl)] = o
        print(f"{'touch' if r == 't' else 'whisking'} neurons [{cyl}]: (a) pooled median proximal {np.nanmedian(o['allp']):+.3f} (n {len(o['allp'])}), "
              f"distal {np.nanmedian(o['alld']):+.3f} (n {len(o['alld'])}), signed-rank on mouse means p = {stats.wilcoxon(o['pm'], o['dm']).pvalue:.3f}; "
              f"(b) grand median of mouse medians {np.median(o['pmed']):+.3f} vs {np.median(o['dmed']):+.3f}, p = {stats.wilcoxon(o['pmed'], o['dmed']).pvalue:.3f}; "
              f"lambda {o['lam']:.1f} um; bin means {np.round(o['y'], 3).tolist()}")
o = DIST[("t", "methods")]
print("Per mouse, touch neurons, mean change proximal and distal:")
print(pd.DataFrame({"mouse": P.ANIMS["touch"], "proximal": np.round(o["pm"], 3), "distal": np.round(o["dm"], 3)}).to_string(index=False))
fig, ax = plt.subplots(figsize=(4, 3))
for r, c in [("t", "C0"), ("w", "C2")]:
    ax.plot(DIST[(r, "methods")]["x"], DIST[(r, "methods")]["y"], "o-", c=c, label=r)
ax.axhline(0, c="k", lw=0.5); ax.set_xlabel("distance to closest ablated neuron (um)"); ax.set_ylabel("mean score change"); ax.legend()

**Result: distance-dependence-touch reproduced.** Under reading (a), proximal -0.133 and distal
-0.056 (-0.132 and -0.056 with the code's cylinder, the paper's values), signed-rank p = 0.012
(0.020), and the length constant is 87.3 um (87.6), all within tolerance. Under reading (b) the
values are -0.103 and -0.019 with p = 0.359, which do not match; the claim holds under (a).

**Result: distance-dependence-whisking-absent reproduced.** Under the code's reading the
proximal and distal whisking changes are -0.004 and -0.023 (0.000 and -0.023 with the code's
cylinder) and the paired test gives p = 0.164 (0.129; the paper reads 0.098 from its figure),
so no distance dependence at the 0.05 level. Per-mouse medians (variant) give p = 0.652.

## behaviour-unchanged

> Ablations did not change whisker movement or behaviour, so behaviour does not account for the
> decline of the touch representation; no mouse showed a significant change in whisker angle,
> angle at touch or curvature change at touch.

Per mouse and ablation, over all pre and all post session files: fraction correct (hits over
hits and errors; trials without a lick are not scored, as the code sums the first four trial
types) and touch bursts per trial. Touches are merged into bursts at a 50 ms interval; the
toolbox method the code calls is not public, so a touch starting less than 50 ms after the
previous touch of the trial ended is taken to join its burst (an assumption; it bears only on
touches per trial). Kolmogorov-Smirnov tests per mouse on the samples (Methods), and on
normalized histograms as the code does (variant).

In [ ]:
brow = []
for (a, abl) in P.ABLATIONS:
    b = P.behaviour(data[a], a, abl)
    row = {"mouse": a, "ablation": abl, "fc_pre": b["pre"]["frac_correct"], "fc_post": b["post"]["frac_correct"],
           "tpt_pre": b["pre"]["touches_per_trial"], "tpt_post": b["post"]["touches_per_trial"],
           "trials_pre": b["pre"]["n_trials"], "trials_post": b["post"]["n_trials"]}
    for v, edges in [("theta", np.arange(-45.5, 91, 1)), ("theta_at_touch", np.arange(-45.5, 91, 1)), ("kappa_at_touch", np.arange(-0.0255, 0.026, 0.001))]:
        x, y = b["pre"][v], b["post"][v]
        x, y = x[~np.isnan(x)], y[~np.isnan(y)]
        k = stats.ks_2samp(x, y)
        row[f"KS_{v}_p"], row[f"KS_{v}_D"], row[f"n_{v}"] = k.pvalue, k.statistic, min(len(x), len(y))
        hx, hy = np.histogram(x, edges)[0], np.histogram(y, edges)[0]
        row[f"KShist_{v}_p"] = stats.ks_2samp(hx / hx.sum(), hy / hy.sum()).pvalue
        row[f"median_shift_{v}"] = np.median(y) - np.median(x)
    brow.append(row)
beh = pd.DataFrame(brow)
print(beh[["mouse", "ablation", "trials_pre", "trials_post", "fc_pre", "fc_post", "tpt_pre", "tpt_post"]].round(3).to_string(index=False))
print()
paper = {("touch", "fc"): 0.863, ("touch", "tpt"): 0.666, ("silent", "fc"): 0.065, ("silent", "tpt"): 0.721, ("whisking", "fc"): 0.710, ("whisking", "tpt"): 0.620}
for abl in ["touch", "silent", "whisking"]:
    s = beh[beh.ablation == abl]
    for v in ["fc", "tpt"]:
        print(f"{abl:8s} {'fraction correct' if v == 'fc' else 'touches/trial':16s}: mean {s[v + '_pre'].mean():.3f} -> {s[v + '_post'].mean():.3f}; "
              f"signed-rank (paired, legend) p = {stats.wilcoxon(s[v + '_pre'], s[v + '_post']).pvalue:.3f}; "
              f"rank-sum (code) p = {stats.mannwhitneyu(s[v + '_pre'], s[v + '_post'], method='exact').pvalue:.3f}; paper {paper[(abl, v)]}")
print()
cols = [c for c in beh.columns if c.startswith("KS_") or c.startswith("KShist_") or c.startswith("median_shift") or c.startswith("n_")]
print(beh[["mouse", "ablation"] + cols].round(4).to_string(index=False))
for v in ["theta", "theta_at_touch", "kappa_at_touch"]:
    print(f"{v}: KS on samples p < 0.05 in {(beh[f'KS_{v}_p'] < 0.05).sum()} of 24 ablations (median D {beh[f'KS_{v}_D'].median():.3f}); "
          f"KS on histograms p < 0.05 in {(beh[f'KShist_{v}_p'] < 0.05).sum()} of 24")

In [ ]:
# trial-type order, by file format (fraction correct takes the first two of the first four types)
for fmt in ["v5", "v7.3"]:
    seen = {}
    for a in sorted(data):
        for dname, fl in data[a]["sess"].items():
            for fn, r in fl.items():
                if r["format"] == fmt:
                    seen.setdefault(tuple(r["trial_type_str"][:6]), []).append(f"an{a}")
    for k, v in seen.items():
        print(f"{fmt}: trialTypeStr[:6] = {list(k)} in {len(v)} files")
# touch properties: keys and lengths against the event edges (the [0::2] rule assumes one value per edge)
lens = []
for a in sorted(data):
    for dname, fl in data[a]["sess"].items():
        for fn, r in fl.items():
            pl = r["touch_prop_lens"]
            lens.append({"mouse": a, "file": fn, "edges": r["n_touch_edges"], "keys": tuple(sorted(pl)),
                         "theta_len": pl.get("thetaAtTouchOnset", -1), "kappa_len": pl.get("kappaMaxAbsOverTouch", -1)})
lens = pd.DataFrame(lens)
print("touch property keys:", lens["keys"].value_counts().to_dict())
print(f"files: {len(lens)}; thetaAtTouchOnset length equal to the number of event edges in {(lens.theta_len == lens.edges).sum()}, "
      f"to half of it in {(lens.theta_len * 2 == lens.edges).sum()}; kappaMaxAbsOverTouch equal to edges in {(lens.kappa_len == lens.edges).sum()}, "
      f"to half in {(lens.kappa_len * 2 == lens.edges).sum()}")
odd = lens[(lens.theta_len != lens.edges) | (lens.kappa_len != lens.edges)]
print(odd.drop(columns="keys").to_string(index=False) if len(odd) else "no file differs")

**Result: not reproduced.** With the paired signed-rank test the legend names, fraction correct
rises after silent ablations (mean 0.697 -> 0.786, p = 0.008) and touches per trial rise after
whisking ablations (p = 0.047); for touch ablations neither changes (p = 0.910 and 0.570). The
KS tests on the samples find a difference in all 24 ablations for whisker angle and in 23 of 24
for angle and curvature at touch, with small effects (median D 0.085, 0.128 and 0.105), as
the large samples make the test sensitive. The code's tests find nothing: the rank-sum tests
give no p below 0.05, and equal the figure's values for fraction correct (0.863, 0.065, 0.710)
and for whisking-ablation touches (0.620); for touches per trial after touch and silent
ablations they give 0.796 and 0.645 against 0.666 and 0.721, as the burst rule here is
assumed. The KS tests on normalized histograms give no p below 0.05. So the paper's statistics come from unpaired tests, not the
paired ones its legend names. The part of the statement about touch ablations (no change in
behaviour that could explain the touch decline) holds under every test run here. The
trial types begin HitL, HitR, ErrL, ErrR in every file of both formats, so fraction correct is
hits over hits and errors; the touch properties have one value per event edge in all 201
files, so taking every other value gives one per touch.

## response-similarity-touch-ablation, response-similarity-example, response-similarity-whisking-ablation

> After touch ablation, the change in Rtouch depended on response similarity to the ablated
> neurons; similar neurons declined and neurons with negative similarity increased, while the
> whisking representation showed no such dependence. In an example touch ablation mouse
> (272761), the slope is -0.44. Ablation of whisking neurons had no similarity-dependent effect
> on either representation.

Members with score above 0.25. Main reading: trial averages of correct left and right trials
concatenated (Methods), Methods cylinder, a line fitted to the neurons' points per mouse, sign
test on the slopes. Variants: the code's trial average (right trials only), the code's
cylinder, and the code's fit to bin means.

In [ ]:
SIM = {}
for abl in ["touch", "whisking"]:
    for ta in ["both", "right"]:
        for a in P.ANIMS[abl]:
            SIM[(abl, ta, a)] = P.similarity(data[a], T[(a, abl)], a, abl, ta)

def sim_summary(abl, r, ta, cyl):
    rows, bms = [], []
    for a in P.ANIMS[abl]:
        d = P.with_cyl(T[(a, abl)], cyl)
        c = SIM[(abl, ta, a)]
        m = P.members(d, r, 0.25) & ~np.isnan(c)
        ds = (d[f"{r}_post"] - d[f"{r}_pre"]).values
        s_pts, s_bin, bm, n, n_in = P.similarity_slopes(c[m], ds[m])
        rows.append({"mouse": a, "slope_points": s_pts, "slope_bins": s_bin, "n": n, "n_in_bins": n_in})
        bms.append(bm)
    return pd.DataFrame(rows), np.array(bms)

SS = {}
for abl in ["touch", "whisking"]:
    for r in ["t", "w"]:
        for ta in ["both", "right"]:
            for cyl in ["methods", "code"]:
                s, bms = sim_summary(abl, r, ta, cyl)
                SS[(abl, r, ta, cyl)] = (s, bms)
                print(f"{abl} ablation, {'touch' if r == 't' else 'whisking'} neurons, trial avg {ta:5s}, {cyl:7s} cylinder: n {s.n.sum():5d} (in bins {s.n_in_bins.sum():5d}); "
                      f"point fit: {(s.slope_points < 0).sum()}/{len(s)} negative, median slope {s.slope_points.median():+.3f}, sign p = {P.sign_test(s.slope_points):.3f}; "
                      f"bin fit: {(s.slope_bins < 0).sum()}/{len(s)} negative, sign p = {P.sign_test(s.slope_bins):.3f}")
print("Paper: touch ablation, touch 2,768 neurons p = 0.039; whisking 1,085 neurons p = 0.508; whisking ablation, touch 1,692 p = 0.453; whisking 935 p = 1")

In [ ]:
# trial averages: files without them, and files whose averages are longer than the 70 frames kept
for a in sorted(data):
    for dname, fl in data[a]["sess"].items():
        for fn, r in sorted(fl.items()):
            if r["hitL"] is None or r["hitR"] is None:
                ks = [k for k in r["feature_keys"] if "TrialAvg" in k or "dffBased" in k or "Hit" in k]
                print(f"an{a} {dname} {fn}: hitL {'none' if r['hitL'] is None else r['hitL'].shape}, hitR {'none' if r['hitR'] is None else r['hitR'].shape}; "
                      f"{len(r['feature_keys'])} feature keys, of which trial-average or dF/F keys: {ks}")
long_ = sorted({(f"an{a}", dname.replace("session_neuropilone_", "").replace("_merged", ""), fn[-19:-16], r["hitL"].shape[0])
                for a in sorted(data) for dname, fl in data[a]["sess"].items() for fn, r in fl.items()
                if r["hitL"] is not None and r["hitL"].shape[0] != 70})
print(f"files whose trial averages are not 70 frames long ({len(long_)}), cut to 70:", long_)
print("whisking ablation mice entering the similarity analysis:",
      {a: int(np.isfinite(SIM[("whisking", "both", a)]).sum()) for a in P.ANIMS["whisking"]})

In [ ]:
xc = np.arange(-1, 1, 0.1) + 0.05
for ta in ["both", "right"]:
    s, bms = SS[("touch", "t", ta, "methods")]
    print(f"touch ablation, touch neurons, trial avg {ta}: mean over mice of bin means, by similarity bin:")
    print(pd.Series(np.nanmean(bms, 0), index=np.round(xc, 2)).dropna().round(3).to_string())
    print(f"  bins with negative similarity and a mean: {int(np.sum(~np.isnan(np.nanmean(bms, 0)[xc < 0])))}; above 0 in {int(np.sum(np.nanmean(bms, 0)[xc < 0] > 0))}")
s, _ = SS[("touch", "t", "both", "methods")]
print(s.round(3).to_string(index=False))
fig, axs = plt.subplots(1, 2, figsize=(8, 3))
for ax, ta in zip(axs, ["both", "right"]):
    _, bms = SS[("touch", "t", ta, "methods")]
    for b in bms:
        ax.plot(xc, b, c="0.7", lw=0.8)
    ax.plot(xc, np.nanmean(bms, 0), c="C0", lw=2); ax.axhline(0, c="k", lw=0.5)
    ax.set_title(f"trial average: {ta}"); ax.set_xlabel("response similarity"); ax.set_ylabel("change in Rtouch")
    print(ta, "plotted range", np.round([np.nanmin(bms), np.nanmax(bms)], 3))

**Result: response-similarity-touch-ablation not reproduced.** Under the claims file's method
(both trial averages concatenated, a line through each mouse's neurons), 7 of 9 touch slopes are
negative (sign test p = 0.180, not below 0.05), and the whisking slopes are positive in 8 of 9
(p = 0.039), a dependence the claim says is absent; the mean bin change is above 0 at negative
similarity, as stated. The paper's numbers are reproduced only with the code's trial average,
which keeps the right-trial average alone: then 8 of 9 touch slopes are negative (p = 0.039)
with 2,768 neurons in bins (code cylinder, bin-mean fit), and the whisking slopes give p = 0.508
with 1,085 neurons, both exactly the paper's values.

In [ ]:
for ta in ["both", "right"]:
    for cyl in ["methods", "code"]:
        s, _ = SS[("touch", "t", ta, cyl)]
        e = s[s.mouse == "272761"].iloc[0]
        print(f"an272761, trial avg {ta:5s}, {cyl:7s} cylinder: slope of the point fit {e.slope_points:+.3f}; slope of the bin-mean fit {e.slope_bins:+.3f}; n {e.n}")
d = P.with_cyl(T[("272761", "touch")], "methods")
c = SIM[("touch", "both", "272761")]
m = P.members(d, "t", 0.25) & ~np.isnan(c)
fig, ax = plt.subplots(figsize=(4, 3))
ax.scatter(c[m], (d.t_post - d.t_pre).values[m], s=6)
ax.set_xlabel("response similarity"); ax.set_ylabel("change in Rtouch"); ax.set_title("an272761")
print("plotted ranges: x", np.round([np.nanmin(c[m]), np.nanmax(c[m])], 2), "y", np.round([np.nanmin((d.t_post - d.t_pre).values[m]), np.nanmax((d.t_post - d.t_pre).values[m])], 2))

**Result: response-similarity-example not reproduced.** With the claim's method the slope of
the point fit in an272761 is -0.393, just outside the tolerance (-0.484 to -0.396). With the
code's right-trial average the point-fit slope is -0.433 (Methods cylinder) or -0.426 (code
cylinder), within tolerance; the code's bin-mean fit gives -0.617 and -0.608.

In [ ]:
for r in ["t", "w"]:
    s, _ = SS[("whisking", r, "both", "methods")]
    print(f"whisking ablation, {'touch' if r == 't' else 'whisking'} neurons:")
    print(s.round(3).to_string(index=False))

**Result: response-similarity-whisking-ablation reproduced.** With the claim's method the sign
tests give p = 1.000 for touch neurons (3 of 6 slopes negative) and p = 0.688 for whisking
neurons (4 of 6), so no similarity dependence at the 0.05 level, with 1,624 and 889 neurons
(within tolerance of 1,692 and 935). Only 6 of the 7 mice enter: the six pre-ablation files
of an278939's whisking ablation (its post-silent directory) hold no trial-averaged dF/F, so no
similarity can be computed there (they hold 27 features, of which psthHitLeftTrialsMatrix and
psthHitRightTrialsMatrix, which no other file uses for this analysis; an278939 enters with 0
neurons). The code's bin-mean fit (variant) gives p = 0.031 for touch
neurons with both trial averages, and the right-trial point fit also gives 0.031.

## ablation-no-off-target-event-rate

> Ablation of silent neurons did not change the calcium event rate of neurons adjacent (10-25
> um) to the ablated cells (0.014 Hz before and after).

As ablation_figure_distance.m (event_rate_plot): neurons more than 10 and less than 25 um from
the closest ablated neuron, leaving out ablated and cylinder-excluded neurons, all
representations; per-mouse median event rate; signed-rank test.

In [ ]:
for cyl in ["methods", "code"]:
    pre, post, n, per = [], [], 0, []
    for a in P.ANIMS["silent"]:
        d = T[(a, "silent")]
        ex = (d[f"prox_{cyl}"] == 1) | (d.is_ablated == 1)
        adj = (d.d_ablated_um > 10) & (d.d_ablated_um < 25) & ~ex
        pre.append(np.nanmedian(d.er_pre[adj])); post.append(np.nanmedian(d.er_post[adj])); n += int(adj.sum())
        per.append((a, int(adj.sum()), round(pre[-1], 4), round(post[-1], 4)))
    print(f"[{cyl}] grand median {np.median(pre):.4f} -> {np.median(post):.4f} Hz (adjusted MAD {P.adj_mad(pre):.4f}, {P.adj_mad(post):.4f}); "
          f"{n} neurons; signed-rank p = {stats.wilcoxon(pre, post).pvalue:.4f}; mice with a lower rate after: {int(np.sum(np.array(post) < np.array(pre)))} of 8")
    print("   per mouse (mouse, n, pre, post):", per)

**Result: reproduced.** Grand median event rate 0.0141 -> 0.0140 Hz with 1,035 adjacent
neurons (1,028 with the code's cylinder, the paper's count), signed-rank p = 0.055, no change at
the 0.05 level (7 of 8 mice have a slightly lower rate after).

## ablation-success-rate

> About two thirds of ablation attempts succeeded; 293 cells were ablated successfully in 22
> sessions in 14 mice with complete logs.

Targets and successes from the curation files (NYU: per-attempt targets and success flags; JRC:
`attempted_ids` and `ablated_ids`). The JRC ablation logs' target ids (`la.ablatedId`, as
ablation_figure_energy.m reads them) are compared with `attempted_ids`.

In [ ]:
srow = []
for (a, abl) in P.ABLATIONS:
    tg, sc, logs = P.targets_and_successes(data[a], a, abl)
    srow.append({"mouse": a, "ablation": abl, "targets": len(tg), "successes": len(sc), "successes_not_targets": len(sc - tg),
                 "log_targets": len(logs) if a not in P.NYU else None, "logs_eq_attempted": (logs == tg) if a not in P.NYU else None})
sr = pd.DataFrame(srow)
print(sr.to_string(index=False))
ok = sr[sr.targets > 0]
print(f"\nAblations with a list of targets: {len(ok)} of 24 (an250220's curation file lists only the ablated ids, and its logs are under dates its curation file does not name)")
print(f"Fraction successful over those: {ok.successes.sum()} / {ok.targets.sum()} = {ok.successes.sum() / ok.targets.sum():.3f}; "
      f"per ablation: median {np.median(ok.successes / ok.targets):.3f}, range {np.min(ok.successes / ok.targets):.2f} to {np.max(ok.successes / ok.targets):.2f}")
rest = sr[~sr.mouse.isin(["250220", "257218"])]
print(f"Successes leaving out an250220 and an257218: {rest.successes.sum()} in {len(rest)} ablations of {rest.mouse.nunique()} mice; all 24 ablations: {sr.successes.sum()}")
print("\nNYU mice: per-attempt log files in the archive against the curation file's targets and attempts")
for (a, abl) in P.ABLATIONS:
    if a not in P.NYU:
        continue
    cur = P.ABLATIONS[(a, abl)][2]
    att = [n for n in data[a]["nyu_attempts"]]
    tg_ids = set()
    n_att = 0
    for c in cur:
        dcur = data[a]["small"][c]
        if "ablationTargetIds" in dcur:
            ids = np.atleast_1d(dcur["ablationTargetIds"])
            n_att += len(ids)
            tg_ids |= set(ids.tolist())
    import re as _re
    log_ids = [int(m.group(1)) for n in att for m in [_re.search(r"_target_(\d+)_attempt_", n)] if m]
    in_tg = sum(i in tg_ids for i in set(log_ids))
    print(f"an{a} {abl:8s}: curation attempts {n_att}, distinct targets {len(tg_ids)}; attempt logs in the archive (all ablations of the mouse) {len(att)}, "
          f"distinct target ids in their names {len(set(log_ids))}, of which in this ablation's targets {in_tg}")
print("NYU totals: attempt logs", sum(len(data[a]["nyu_attempts"]) for a in sorted(P.NYU)))

**Result: reproduced.** Over the 23 ablations with a list of targets, 320 of 474 targeted neurons
were ablated (0.675, within 0.667 ± 0.067). Leaving out the two mice the legend names, 300
neurons were ablated successfully in 22 ablations of 14 mice, within tolerance of 293; the
legend also leaves out 7 other ablations it does not name, so this count cannot be matched
exactly. In the NYU mice the 510 per-attempt logs are the attempts of the curation files: each
mouse has as many logs as curation-file attempts over its ablations (for example 30 + 61 = 91
in an275798), and their target ids are the curation files' targets (the 279 NYU targets are
distinct neurons, several attempted more than once).

## per-mouse-table

> Extended Data Table 1 lists, for each of the 24 ablations, the mouse's neuron count, the
> ablation type, the number ablated, the touch and whisking neuron counts and the change in
> median Rtouch and Rwhisking.

Counts match within 10%, changes within 0.005. The touch and whisking counts are compared under
reading (a) of representation-sizes (score above 0.1, first pre-ablation data of the ablation),
reading (b), and the members entering the medians (the union of the pre and post periods, as in
get_restricted_neurons.m), with each cylinder. The cell count of a second ablation is the
mouse's (the table gives it once).

In [ ]:
prow = []
for e in EDT1.itertuples():
    for cyl in ["methods", "code"]:
        d = P.with_cyl(T[(e.mouse, e.ablation)], cyl)
        tpre, tpost, nt = P.mouse_medians(d, "t")
        wpre, wpost, nw = P.mouse_medians(d, "w")
        cells = e.cells if not pd.isna(e.cells) else None
        fa = next(k for k in T if k[0] == e.mouse and k[1] in FIRST and e.mouse in FIRST[k[1]])
        prow.append({"mouse": e.mouse, "ablation": e.ablation, "cyl": cyl, "cells": len(T[fa]), "EDT1_cells": cells,
                     "ablated": int(d.is_ablated.sum()), "EDT1_ablated": e.ablated,
                     "touch_members": nt, "touch_a": P.class_counts(d, "t", "a")[0], "touch_b": P.class_counts(d, "t", "b")[0], "EDT1_touch": e.touch,
                     "whisk_members": nw, "whisk_a": P.class_counts(d, "w", "a")[0], "whisk_b": P.class_counts(d, "w", "b")[0], "EDT1_whisk": e.whisking,
                     "dRt": tpost - tpre, "EDT1_dRt": e.dRtouch, "dRw": wpost - wpre, "EDT1_dRw": e.dRwhisking})
pt = pd.DataFrame(prow)
within = lambda x, y: (np.abs(x - y) <= 0.1 * np.abs(y))
for cyl in ["methods", "code"]:
    s = pt[pt.cyl == cyl]
    cm = (s.EDT1_cells.isna() | (s.cells == s.EDT1_cells))
    am = s.ablated == s.EDT1_ablated
    dt, dw = np.abs(s.dRt - s.EDT1_dRt) <= 0.005, np.abs(s.dRw - s.EDT1_dRw) <= 0.005
    allm = cm & am & dt & dw
    print(f"[{cyl}] cell count {cm.sum()}/24, ablated {am.sum()}/24, dRtouch {dt.sum()}/24, dRwhisking {dw.sum()}/24, all four {allm.sum()}/24; "
          f"touch count within 10%: members {within(s.touch_members, s.EDT1_touch).sum()}, (a) {within(s.touch_a, s.EDT1_touch).sum()}, (b) {within(s.touch_b, s.EDT1_touch).sum()}; "
          f"equal: members {(s.touch_members == s.EDT1_touch).sum()}; whisking within 10%: members {within(s.whisk_members, s.EDT1_whisk).sum()}, "
          f"(a) {within(s.whisk_a, s.EDT1_whisk).sum()}, (b) {within(s.whisk_b, s.EDT1_whisk).sum()}; equal: members {(s.whisk_members == s.EDT1_whisk).sum()}")
s = pt[pt.cyl == "methods"].copy()
print(s.drop(columns="cyl").round(3).to_string(index=False))
print("\nRows where a change differs by more than 0.005 (Methods cylinder):")
print(s[(np.abs(s.dRt - s.EDT1_dRt) > 0.005) | (np.abs(s.dRw - s.EDT1_dRw) > 0.005)][["mouse", "ablation", "dRt", "EDT1_dRt", "dRw", "EDT1_dRw"]].round(3).to_string(index=False))

**Result: partially reproduced.** Under the main reading (the Methods cylinder) 23 of 24 rows
match in cell count, ablated count and both changes; in an275801's touch ablation the change in
Rtouch is -0.071 against the table's -0.065. With the code's cylinder (variant) all 24 rows
match. The table's touch and whisking counts are the members that enter the medians under the
code's cylinder (21 and 22 of 24 rows equal, all within 10%), not the counts of reading (a) or
(b) (10 and 1 rows within 10% for touch), so the table was made with the code's rule.

## Network model claims

The model outputs were not deposited (claims file, `data: elsewhere`), so the network of the
Methods ("Network model") is rebuilt here in numpy and numba (helper `peron2020_model.py`, whose
docstring gives the source of every parameter). The paper used Brian2; this is a
reimplementation of the same equations and parameters, not the paper's code.

What the Methods leave open, and the choice made for each (each an assumption, recorded in the
attempt):

- The size of the background PSP. The Methods give its frequencies (5,000 and 2,000 Hz) and say
  they were "selected so that the tonic firing rate of these populations were approximately 0.5
  Hz for excitatory neurons and 10 Hz for inhibitory neurons". One PSP per population is set
  here to give those rates at Pconn 0.2 without stimulus (first cell below).
- The sign of inhibitory PSPs: -1 mV ("All connections had the same strength, which was chosen
  so that the resulting post-synaptic potential (PSP) is 1 mV").
- The stimulus amplitude is its peak current; its value comes from the readings of the claims
  file's opening comment, (a) fraction of excitatory neurons with score above 0.1 equal to 0.108,
  (b) smallest Kolmogorov-Smirnov distance to the pooled pre-ablation Rtouch of the touch ablation
  mice, (c) fraction equal to 0.17. Reading (b) sets a strength only where its smallest distance
  lies at a positive strength inside the sweep, which reaches down to no input.
- Which neurons enter "all neurons that met the 0.1 encoding score criteria": excitatory
  neurons only (main) and all neurons of the network, inhibitory ones included (variant). The
  Methods match the input on the "distributions of excitatory neuron encoding scores", ablate
  "the 20 excitatory neurons with the highest encoding score", and the in vivo data the model
  stands for are of excitatory neurons (Ai162 x Slc17a7-IRES2-cre mice; Emx1-IRES-cre mice,
  whose mCherry marks "only ... nuclei of excitatory neurons"; Methods, Mice). The paper's
  counts for Fig. 3b (3,466 and 3,779 neurons above 0.25 over 30 networks) are also nearer the
  excitatory counts here than those of all neurons (model-response-similarity cell). An earlier
  revision of this notebook counted all neurons; the outcomes that the change moves are proposed
  in the attempt record for a further verification round.
- The repeated run after ablation uses the same background and initial state as the run before
  (Brian2's store and restore); the 0-ablation networks of Extended Data Fig. 4 therefore show
  no change.

Every network is seeded by its index (0 to 29), so each claim's numbers do not depend on the
order of the cells.

In [ ]:
import peron2020_model as PM

t_model = time.time()
bg_grid = [(be, bi) for be in [0.37, 0.39, 0.41] for bi in [1.3, 1.4, 1.5]]
bgr = PM.run_jobs([dict(kind="pre_only", seeds=[800], pconn=0.2, strength=0.0, bg=g, duration=4000.0) for g in bg_grid])
print("Background PSP (mV, excitatory and inhibitory targets) and tonic rates at Pconn 0.2 without stimulus, one network, 4 s:")
for g, d in zip(bg_grid, bgr):
    print(f"  {g}: excitatory {d['rate_pre'][:PM.NEXC].mean():.2f} Hz, inhibitory {d['rate_pre'][PM.NEXC:].mean():.2f} Hz")
print("Used:", PM.BG_PSP, "(Methods: about 0.5 Hz and 10 Hz)")
stim10, n10 = PM.stimulus(10.0)
stim50, n50 = PM.stimulus(50.0)
on = np.nonzero(np.diff(np.r_[0, (stim10 > 0).astype(int)]) == 1)[0]
w10 = np.sum(stim10[on[0]:on[0] + 400] > 0) * PM.DT
half = stim10[on[0]:on[0] + 400] >= 0.5
print(f"Stimulus: {n10} presentations in 20 s, first onset {on[0] * PM.DT:.0f} ms, every {np.diff(on).mean() * PM.DT:.0f} ms; touch waveform "
      f"{w10:.1f} ms long, peak at {np.argmax(stim10[on[0]:on[0] + 400]) * PM.DT:.1f} ms, full width at half maximum {half.sum() * PM.DT:.1f} ms (Methods 12.8 ms); "
      f"slow waveform peak at {np.argmax(stim50[on[0]:on[0] + 2000]) * PM.DT:.1f} ms ({n50} presentations)")
print(f"Kick sizes (equation 7) for a 1 mV PSP: E->E {PM.psp_to_kick(1, 30, 2):.3f}, E->I {PM.psp_to_kick(1, 10, 2):.3f}, "
      f"I->E {PM.psp_to_kick(1, 30, 3):.3f}, I->I {PM.psp_to_kick(1, 10, 3):.3f}")
_b = PM.Batch([0], 0.4)
_g = _b.groups
_pre_g, _post_g = _g[_b.pre], _g[_b.post]
print("Connection probability by block (pre -> post), network 0 at Pconn 0.4:",
      {f"{'SEI'[i]}->{'SEI'[j]}": round(float(((_pre_g == i) & (_post_g == j)).sum() / ((_g == i).sum() * ((_g == j).sum() - (i == j)))), 3)
       for i in range(3) for j in range(3)})
print(f"Delays {(_b.delay.min() * PM.DT):.1f} to {(_b.delay.max() * PM.DT):.1f} ms; Vth - Vr {_b.theta.min():.1f} to {_b.theta.max():.1f} mV")
del _b

### Calibration of the input strength

Sweeps of the peak input current, two networks per strength (seeds 900 and 901), for the touch
input (peak 10 ms) and the slow input (peak 50 ms), at Pconn 0.2 and 0.4. For each strength,
the fraction of excitatory neurons with score above 0.1 and the Kolmogorov-Smirnov distance
between the excitatory scores and the in vivo distribution (pre-ablation Rtouch of every neuron
of the 9 touch ablation mice). Reading (a) interpolates linearly to the fraction 0.108, reading
(c) to 0.17, and reading (b) takes the strength of the grid with the smallest distance, set
only when that strength is positive and inside the grid. Each sweep starts at no input (strength
0), so that a minimum at the weakest strength tried can be told from one at no input.

In [ ]:
SW = {(0.2, 10.0): [0, 25, 50, 75, 100, 125, 150, 175, 200, 225, 250, 300, 400], (0.4, 10.0): [0, 3, 6, 9, 12, 15, 20, 25, 30, 35, 40, 50],
      (0.2, 50.0): [0, 25, 50, 70, 90, 110, 140, 200], (0.4, 50.0): [0, 1, 2, 3, 5, 7, 9, 12, 15]}
sw_jobs = [dict(kind="pre_only", seeds=[900 + k], pconn=pc, strength=float(s), peak_ms=pk) for (pc, pk), ss in SW.items() for s in ss for k in range(2)]
sw = PM.run_jobs(sw_jobs)
invivo = np.concatenate([T[(a, "touch")].t_pre.dropna().values for a in P.ANIMS["touch"]])
print(f"In vivo reference: {len(invivo)} neurons, fraction above 0.1 {np.mean(invivo > 0.1):.3f}, median {np.median(invivo):.3f}, "
      f"range {invivo.min():.3f} to {invivo.max():.3f}, fraction exactly 0 {np.mean(invivo == 0):.3f}")
cal = []
for (pc, pk), ss in SW.items():
    for s in ss:
        ds = [d for d in sw if d["pconn"] == pc and d["peak"] == pk and d["strength"] == s]
        e = np.concatenate([d["pre"][:PM.NEXC] for d in ds])
        cal.append({"pconn": pc, "peak_ms": pk, "strength": s, "frac_exc_0.1": np.mean(e > 0.1), "frac_exc_zero": np.mean(e == 0), "frac_S_0.1": np.mean(np.concatenate([d["pre"][:PM.NS] for d in ds]) > 0.1),
                    "median_exc_above_0.1": np.median(e[e > 0.1]) if (e > 0.1).any() else np.nan, "KS": stats.ks_2samp(e, invivo).statistic,
                    "rate_exc_Hz": np.mean([d["rate_pre"][:PM.NEXC].mean() for d in ds]), "rate_inh_Hz": np.mean([d["rate_pre"][PM.NEXC:].mean() for d in ds])})
cal = pd.DataFrame(cal)
print(cal.round(3).to_string(index=False))

def cross(x, f, target):
    """First crossing of ``target`` by f(x), linear interpolation between grid points."""
    for i in range(1, len(x)):
        if f[i - 1] < target <= f[i]:
            return float(x[i - 1] + (target - f[i - 1]) * (x[i] - x[i - 1]) / (f[i] - f[i - 1]))
    return np.nan

STR = {}
for (pc, pk), g in cal.groupby(["pconn", "peak_ms"]):
    x, f = g.strength.values, g["frac_exc_0.1"].values
    STR[("a", pc, pk)] = cross(x, f, 0.108)
    STR[("c", pc, pk)] = cross(x, f, 0.17)
    k = int(np.argmin(g.KS.values))
    STR[("b", pc, pk)] = float(x[k]) if 0 < k < len(x) - 1 else np.nan  # a minimum at no input or at the strongest input sets none
    print(f"Pconn {pc}, peak {pk:.0f} ms: (a) {STR[('a', pc, pk)]:.1f}; (b) {STR[('b', pc, pk)]:.1f} (smallest KS {g.KS.min():.3f} at strength {x[k]:g}; "
          f"next smallest {np.sort(g.KS.values)[1]:.3f}); "
          f"(c) {STR[('c', pc, pk)]:.1f} (largest fraction reached {f.max():.3f})")
print("\nAmplification (inverse of the matched input strength, normalized to Pconn 0.2), touch input:")
AMP = {}
for rd in ["a", "b", "c"]:
    s2, s4 = STR[(rd, 0.2, 10.0)], STR[(rd, 0.4, 10.0)]
    AMP[rd] = s2 / s4
    print(f"  reading ({rd}): strength {s2:.1f} at Pconn 0.2 and {s4:.1f} at 0.4; amplification at 0.4 = {AMP[rd]:.2f}; "
          f"strength at 0.4 below that at 0.2: {s4 < s2 if not np.isnan(s2 + s4) else 'not set'}")
    for p in [0.25, 0.3, 0.35, 0.44]:
        sp = s2 + (p - 0.2) / 0.2 * (s4 - s2)
        print(f"     interpolated strength at Pconn {p}: {sp:.1f}" + (f", amplification {s2 / sp:.2f}" if sp > 0 else ", not positive"))

### Ablation runs

For each reading whose strengths could be set, 30 networks (seeds 0 to 29) at Pconn 0.2 (the 20
strongest excitatory encoders ablated) and at Pconn 0.4 (0, 10, 20 and 50 ablated, Extended Data
Fig. 4), with the touch input; and with the slow input at both Pconn, under (a) the touch input
strength and (b) a strength set anew by the same rule from the slow-input sweep.

In [ ]:
RUNS = {}
def ckey(pc, s, pk, sizes):
    return (pc, round(float(s), 3), pk, tuple(sizes))

conds = []
for rd in ["a", "b", "c"]:
    if np.isnan(STR[(rd, 0.2, 10.0)]) or np.isnan(STR[(rd, 0.4, 10.0)]):
        print(f"reading ({rd}): no strength, not run")
        continue
    conds += [ckey(0.2, STR[(rd, 0.2, 10.0)], 10.0, (20,)), ckey(0.4, STR[(rd, 0.4, 10.0)], 10.0, (0, 10, 20, 50))]
    for pc in [0.2, 0.4]:
        conds += [ckey(pc, STR[(rd, pc, 10.0)], 50.0, (20,))]
        if not np.isnan(STR[(rd, pc, 50.0)]):
            conds += [ckey(pc, STR[(rd, pc, 50.0)], 50.0, (20,))]
conds = list(dict.fromkeys(conds))
NNET = 30
jobs = [dict(kind="condition", seeds=[sd], pconn=c[0], strength=c[1], peak_ms=c[2], sizes=c[3], run_seed=sd,
             with_similarity=(c[2] == 10.0)) for c in conds for sd in range(NNET)]
t0 = time.time()
out = PM.run_jobs(jobs)
for i, c in enumerate(conds):
    RUNS[c] = out[NNET * i:NNET * (i + 1)]
print(f"{len(conds)} conditions, {len(jobs)} networks, {sum(1 + sum(s > 0 for s in c[3]) for c in conds) * NNET} runs of 20 s, {time.time() - t0:.0f} s")
for c in conds:
    rs = RUNS[c]
    print(f"  Pconn {c[0]}, strength {c[1]}, peak {c[2]:.0f} ms, sizes {c[3]}: rates before ablation, excitatory {np.mean([d['rate_pre'][:PM.NEXC].mean() for d in rs]):.2f} Hz, "
          f"inhibitory {np.mean([d['rate_pre'][PM.NEXC:].mean() for d in rs]):.2f} Hz; fraction of excitatory neurons above 0.1 "
          f"{np.mean([np.mean(d['pre'][:PM.NEXC] > 0.1) for d in rs]):.3f}")

def summarize(c, size=20, pop="exc", thresh=0.1):
    e = pd.DataFrame(PM.effect(RUNS[c], size, thresh, pop))
    e["change"] = e.post - e.pre
    p = stats.wilcoxon(e.pre, e.post).pvalue if (e.change != 0).any() else np.nan
    return e, p

def report(c, label, size=20):
    res = {}
    for pop in ["exc", "all"]:
        e, p = summarize(c, size, pop)
        res[pop] = (e, p)
        print(f"{label}, {'all neurons (variant)' if pop == 'all' else 'excitatory only (main)'}: grand median {e.pre.median():.3f} -> {e.post.median():.3f} "
              f"(adjusted MAD {P.adj_mad(e.pre):.3f}, {P.adj_mad(e.post):.3f}); signed-rank p = {p:.2g}; networks with an increase "
              f"{(e.change > 0).sum()} of {len(e)}; neurons per network {e.n.median():.0f} (range {e.n.min()} to {e.n.max()})")
    return res

### model-amplification-increases

> In the L2/3 network model, the sensory input needed to match in vivo responses declines as
> subnetwork connectivity rises, so amplification (network output over input, normalized to
> Pconn 0.2) increases with subnetwork connectivity.

The matched strengths and amplification under each reading are printed in the calibration cell
above.

**Result: reproduced, resting on assumptions.** Under calibration (a) the matched strength is
218.0 at Pconn 0.2 and 28.2 at 0.4, so normalized amplification at 0.4 is 7.74, and the strength
at 0.4 is below that at 0.2. Reading (a) is weakly set at Pconn 0.2: the fraction above 0.1 moves
only from 0.106 to 0.109 between strengths 200 and 225, with two networks per strength, which
bears on how large the amplification is, not on its direction. Reading (b) sets no strength at
Pconn 0.4: with the sweep extended down to no input, the smallest KS distance for the touch input
is at strength 0 (0.514), and the distance hardly depends on the input (over the
whole sweep, 0.514 to 0.582), because 0.66 to 0.75 of the model's excitatory neurons score
exactly 0 at every strength while almost no in vivo neuron
does (fraction 0.000 to three decimals). At Pconn 0.2 its minimum is at 75 (0.468, with
0.469 and 0.470 at 100 and 125). Reading (c), the fraction 0.17, cannot be met: the largest
fraction of excitatory neurons above 0.1 in any sweep is 0.119, as nearly only the 200
input-recipient neurons (11.8% of the excitatory population) reach it. The claim holds under the
one reading that sets both strengths. Amplification here is far larger than the paper's Fig. 1c.

### model-equal-connectivity-ablation-increases, model-increased-connectivity-ablation-decreases

> In networks with equal subnetwork connectivity (Pconn 0.2), ablating the strongest encoders
> raises the encoding scores of spared neurons, from 0.237 to 0.274, owing to reduced feedback
> inhibition. In networks with increased subnetwork connectivity (Pconn 0.4), ablation lowers
> the encoding scores of spared neurons, from 0.243 to 0.143, because amplification is reduced.

In [ ]:
MODEL = {}
for rd in ["a", "b", "c"]:
    if np.isnan(STR[(rd, 0.2, 10.0)]) or np.isnan(STR[(rd, 0.4, 10.0)]):
        continue
    for pc, sizes in [(0.2, (20,)), (0.4, (0, 10, 20, 50))]:
        c = ckey(pc, STR[(rd, pc, 10.0)], 10.0, sizes)
        MODEL[(rd, pc)] = report(c, f"reading ({rd}), Pconn {pc}, touch input, 20 ablated")
print("Paper: Pconn 0.2, 0.237 -> 0.274 (P < 0.001); Pconn 0.4, 0.243 -> 0.143 (P < 0.001); n = 30 networks")
e, _ = MODEL[("a", 0.2)]["exc"]
print("\nReading (a), Pconn 0.2, excitatory neurons, per network:")
print(e.round(3).to_string(index=False))

**Result: model-equal-connectivity-ablation-increases not reproduced as verified; partially
reproduced under the revised population (proposed), resting on assumptions.** At Pconn 0.2 under
calibration (a), the spared excitatory neurons (above 0.1 before or after, ablated ones left
out) go from a grand median of 0.546 to 0.562 (signed-rank p = 1.9e-09, 30 of 30 networks
increase), the claimed increase at p < 0.001, with both values far outside 0.237 ± 0.024 and
0.274 ± 0.027. Counting all neurons (variant), the scores fall, 0.287 to 0.279 (p = 6.3e-05, 4
of 30 increase), and 0.287 is outside tolerance. The verified outcome, not reproduced, was set
with all neurons counted; under the revised choice the effect is present and the values are
not, which is partially reproduced, as for the Pconn 0.4 claim. The change is proposed for a
further verification round.

**Result: model-increased-connectivity-ablation-decreases partially reproduced, resting on
assumptions.** At Pconn 0.4 ablation lowers the spared neurons' scores under both populations:
0.356 to 0.231 (excitatory, p = 3.9e-07, 4 of 30 increase) and 0.282 to 0.096 (all neurons, p =
1.9e-08). Neither gives both grand medians within 0.243 ± 0.024 and 0.143 ± 0.014.

### model-all-or-none-robust

> Subnetwork connectivity above 0.4 gives all-or-none responses, a transient input driving the
> subnetwork into persistent activity, and such networks keep their all-or-none response after
> ablation of 10% of the subnetwork.

The criterion sets the input at Pconn 0.44 to twice the strength extrapolated from the two
matched ones (Methods). The extrapolated strengths are printed in the calibration cell above.

**Result: not tested (blocked, methods underspecified).** Linear extrapolation of the two matched
strengths to Pconn 0.44 gives -9.8 under calibration (a), and readings (b) and (c) set no
strength at Pconn 0.4, so twice the extrapolated strength, the input the Methods use in this
regime, is not a positive current here. The negative value follows from this attempt's
calibration, whose amplification at Pconn 0.4 (7.7) is far above the paper's Fig. 1c. The
Methods also give neither the matched strengths nor the strength and length of the reset pulse
to the inhibitory population, and the pulse they place 300 ms after each onset falls on the
next onset at the 300 ms interval of the claims file. No run was made.

### model-more-ablation-larger-decline

> In the model with Pconn 0.4, ablating more neurons (top 10, 20 or 50 encoders) gives larger
> declines in encoding score, the change correlating with the net encoding score ablated
> (R = -0.65).

In [ ]:
for rd in ["a", "b", "c"]:
    if (rd, 0.4) not in MODEL:
        continue
    c = ckey(0.4, STR[(rd, 0.4, 10.0)], 10.0, (0, 10, 20, 50))
    for pop in ["exc", "all"]:
        rows = []
        for size in (0, 10, 20, 50):
            e, _ = summarize(c, size, pop)
            rows.append(e.assign(size=size))
        a_ = pd.concat(rows)
        med = a_.groupby("size").change.median()
        for lab, sub in [("with the 0-ablation networks", a_), ("without them", a_[a_["size"] > 0])]:
            sub = sub.dropna(subset=["change"])  # networks with no spared neuron above 0.1 have no median
            r_ = stats.pearsonr(sub.net_ablated, sub.change)
            print(f"reading ({rd}), {'all neurons (variant)' if pop == 'all' else 'excitatory only (main)'}, {lab}: R = {r_.statistic:+.3f}, p = {r_.pvalue:.2g}, n = {len(sub)}")
        print(f"   median change by size {med.round(4).to_dict()}; more negative at each larger size: {bool(np.all(np.diff(med.values) < 0))}; "
              f"median net score ablated by size {a_.groupby('size').net_ablated.median().round(2).to_dict()}")
print("Paper: R = -0.65, P < 0.01")

**Result: model-more-ablation-larger-decline partially reproduced as verified; reproduced under
the revised population (proposed), resting on assumptions.** Under calibration (a), with
excitatory neurons, the median change falls with each larger ablation (0, -0.026, -0.123 and
-0.209 for 0, 10, 20 and 50 ablated), and the change correlates with the net score ablated, R =
-0.590 with the 0-ablation networks (p = 1.3e-12, n = 120), within -0.65 ± 0.065, and -0.456
without them (p = 6.3e-06). The criterion asks for R within tolerance under one of the two
readings and a larger decline at each size, so it is met. With all neurons (variant) R is
-0.726 and -0.558, both outside the tolerance. The verified outcome, partially reproduced, was
set with all neurons counted; the change to reproduced is proposed for a further verification
round. Calibration (b) gave the runs of the earlier revision (R = -0.774 and -0.678 with all
neurons); with its sweep extended to no input it sets no strength at Pconn 0.4 (calibration
cell), so it is not run.

### model-slow-input-sensitive

> Networks with Pconn 0.4 given slower, whisking-like input (peak at 50 ms) are also sensitive to
> ablation, encoding scores falling from 0.143 to 0.104; with equal subnetwork connectivity
> (Pconn 0.2) the slow-input representation is not degraded.

In [ ]:
for rd in ["a", "b", "c"]:
    if np.isnan(STR[(rd, 0.2, 10.0)]) or np.isnan(STR[(rd, 0.4, 10.0)]):
        continue
    for sl, pkey in [("(a) touch strength", 10.0), ("(b) strength set anew", 50.0)]:
        for pc in [0.2, 0.4]:
            s = STR[(rd, pc, pkey)]
            if np.isnan(s):
                print(f"calibration ({rd}), slow input {sl}, Pconn {pc}: no strength")
                continue
            report(ckey(pc, s, 50.0, (20,)), f"calibration ({rd}), slow input {sl}, Pconn {pc}, strength {s:.1f}")
print("Paper: Pconn 0.4, 0.143 -> 0.104 (P < 0.001, n = 30); Pconn 0.2, not degraded (no number given)")

**Result: model-slow-input-sensitive partially reproduced, resting on assumptions.** With the
slow-input strength set anew under calibration (a) (6.6 at Pconn 0.4, 87.3 at 0.2), the score of
the excitatory neurons at Pconn 0.4 falls from 0.314 to 0.104 (p = 3.9e-07, 3 of 30 networks
increase); the post value is within 0.104 ± 0.010 and the pre value is outside 0.143 ± 0.014. At
Pconn 0.2 it rises, 0.803 to 0.850 (p = 1.9e-09, 30 of 30), so it is not degraded, as the claim
says. With the touch-input strength (slow reading (a)) the score does not fall at Pconn 0.4
(0.952 to 0.947, p = 0.16). All neurons (variant) give 0.269 to 0.088 at Pconn 0.4 and no change
at 0.2 (0.424 to 0.424, p = 0.21) with the strength set anew. No reading gives both grand
medians within tolerance, so the effect is present and the values are not. Calibration (b) sets
no strength for the slow input at Pconn 0.4 (calibration cell).

### model-response-similarity

> In networks with increased subnetwork connectivity, neurons whose activity is more similar to
> the mean activity of the ablated neurons lose the most encoding score after ablation; with
> equal connectivity the relationship is absent.

Neurons with score above 0.25 before or after ablation (ablated neurons left out); per network,
a line through the neurons' change in score against their similarity to the mean rate of the 20
ablated neurons in the run before ablation (Methods), and a line through the means of 0.1-wide
similarity bins as a variant (`similarity_slopes` of the data analysis).

In [ ]:
for rd in ["a", "b", "c"]:
    if np.isnan(STR[(rd, 0.2, 10.0)]) or np.isnan(STR[(rd, 0.4, 10.0)]):
        continue
    for pc, sizes in [(0.2, (20,)), (0.4, (0, 10, 20, 50))]:
        rs = RUNS[ckey(pc, STR[(rd, pc, 10.0)], 10.0, sizes)]
        for pop in ["exc", "all"]:
            sp, sb, n_ = [], [], 0
            for d in rs:
                m = PM.spared(d, 20, 0.25, pop)
                x, y = d["sim"][20][m], (d["post"][20] - d["pre"])[m]
                a1, a2, _, n1, _ = P.similarity_slopes(x, y)
                sp.append(a1); sb.append(a2); n_ += n1
            sp, sb = np.array(sp), np.array(sb)
            print(f"reading ({rd}), Pconn {pc}, {'all neurons (variant)' if pop == 'all' else 'excitatory only (main)'}: {n_} neurons; point fit {np.sum(sp < 0)}/{np.sum(~np.isnan(sp))} "
                  f"negative, median slope {np.nanmedian(sp):+.3f}, sign p = {P.sign_test(sp):.2g}; bin-mean fit {np.sum(sb < 0)}/{np.sum(~np.isnan(sb))} negative, "
                  f"sign p = {P.sign_test(sb):.2g}")
print("Paper: Pconn 0.4, P < 0.001 (3,466 neurons); Pconn 0.2, P = 0.2 (3,779 neurons); 30 networks each")
print(f"Model section: {(time.time() - t_model) / 60:.1f} min")

**Result: model-response-similarity not reproduced, resting on assumptions.** Under calibration
(a), with excitatory neurons, the per-network slopes of the change in score against response
similarity are negative at Pconn 0.2 (30 of 30, sign test p = 1.9e-09) and show no dependence at
Pconn 0.4 (11 of 30 negative, p = 0.2), the reverse of the claim, which needs negative slopes at
0.4 and none at 0.2. All neurons (variant) give positive slopes at Pconn 0.4 (6 of 30 negative,
p = 0.0014) and mostly positive ones at 0.2 (8 of 30 negative, p = 0.016). The bin-mean fit
leans the same way. The excitatory counts (4,327 at Pconn 0.2 and 4,058 at 0.4) are nearer the
paper's 3,779 and 3,466 than the counts of all neurons (7,815 and 7,078).

### What the model section shows

The rebuilt network reproduces the direction of the paper's results at increased connectivity
(amplification grows with Pconn; ablation lowers scores at Pconn 0.4, more for larger
ablations, for touch and for slow input) and, with excitatory neurons counted, the rise in
scores at equal connectivity, but not the paper's numbers, nor the response-similarity result.
The choices the outcomes rest on are those listed at the top of the section (background PSP,
inhibitory PSP, stimulus amplitude, which neurons are counted) and the calibration: (a) is the
only reading that sets both matched strengths, as (b)'s smallest distance lies at no input at
Pconn 0.4 and (c) cannot be met. Under the calibration the background gives 0.51 Hz and 10.35
Hz without stimulus, as the Methods ask, but at Pconn 0.4 the stimulus raises the excitatory
rate to 2.78 Hz. The median score of the excitatory neurons above 0.1 is 0.553 at strength 200
and 0.578 at 225 (calibration sweep, Pconn 0.2), and the grand median of the spared excitatory
neurons is 0.546, about 2.3 times the paper's 0.237. A failure here may be a failure of this
reimplementation, not of the paper's model.

## Summary

| Claim | Outcome | Note |
|---|---|---|
| model-amplification-increases | Reproduced (assumed) | Matched strength 218 at Pconn 0.2, 28.2 at 0.4 (calibration a); readings (b) and (c) set no strength at 0.4 |
| model-equal-connectivity-ablation-increases | Not reproduced (assumed); partially reproduced proposed | Excitatory neurons rise, 0.546 -> 0.562, p = 1.9e-09; values outside tolerance; all neurons fall, 0.287 -> 0.279 |
| model-increased-connectivity-ablation-decreases | Partially reproduced (assumed) | 0.356 -> 0.231 (excitatory), p < 0.001; values outside tolerance |
| model-all-or-none-robust | Blocked | Extrapolated input strength at Pconn 0.44 is negative; reset pulse not specified |
| model-more-ablation-larger-decline | Partially reproduced (assumed); reproduced proposed | Decline grows with size; R = -0.590 with the 0-ablation networks (excitatory), within -0.65 ± 0.065; all neurons -0.726 / -0.558 |
| model-slow-input-sensitive | Partially reproduced (assumed) | 0.314 -> 0.104 at Pconn 0.4 (excitatory), no decline at 0.2; pre value outside tolerance |
| model-response-similarity | Not reproduced (assumed) | Excitatory: slopes negative at Pconn 0.2 (p = 1.9e-09), none at 0.4 (p = 0.2); all neurons: positive at 0.4 |
| neurons-per-mouse | Reproduced | 8,126 ± 2,436; all 16 counts equal Extended Data Table 1 |
| representation-sizes | Reproduced | Reading (a), score above 0.1 |
| touch-ablation-size | Partially reproduced | 16.8 ± 12.8 and 5.85%; percentile 79.3 ± 27.1 by the criterion's rank rule (70.3 ± 35.0 by the code's) |
| touch-ablation-reduces-touch | Reproduced | 0.123 -> 0.100, p = 0.004 |
| touch-ablation-touch-count | Reproduced | 932 -> 716, reading (a) |
| touch-ablation-whisking-unchanged | Reproduced | p = 0.820 |
| ablation-size-correlation | Reproduced | R = -0.790 and -0.761 |
| distance-dependence-touch | Reproduced | -0.133 / -0.056, lambda 87 um, reading (a) |
| distance-dependence-whisking-absent | Reproduced | p = 0.164 |
| behaviour-unchanged | Not reproduced | Paired tests find changes after silent and whisking ablations; the paper's P values are rank-sum tests |
| whisking-ablation-size | Partially reproduced | 12.7 ± 5.7; 4.43% against 4 ± 0.4; percentile 75.4 ± 31.8 by the rank rule |
| whisking-ablation-no-effect-touch | Reproduced | p = 0.109 |
| whisking-ablation-no-effect-whisking | Reproduced | p = 0.938 (0.812 with the code's cylinder) |
| silent-ablation-size | Reproduced | 16.3 ± 2.6 |
| silent-ablation-no-effect-touch | Reproduced | p = 0.383 |
| silent-ablation-no-effect-whisking | Reproduced | p = 0.844 |
| ablation-type-specificity | Reproduced | Rank-sum tests |
| response-similarity-example | Not reproduced | Slope -0.393 against -0.44 ± 0.044; -0.433 with the code's trial average |
| response-similarity-touch-ablation | Not reproduced | Sign test p = 0.180 (touch) and 0.039 (whisking) with the Methods trial average; the paper's values come from the code's right-trial average |
| response-similarity-whisking-ablation | Reproduced | p = 1.000 and 0.688 |
| ablation-no-off-target-event-rate | Reproduced | 0.014 Hz before and after, p = 0.055 |
| ablation-success-rate | Reproduced | 0.675 of targets; 300 cells |
| glial-reaction-local | Blocked | Histology not in the data set |
| per-mouse-table | Partially reproduced | 23 of 24 rows with the Methods cylinder; all 24 with the code's |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The encoding scores, shuffle scores, event rates
and trial averages are the authors' own, read from the session files; nothing is recomputed from
fluorescence. The Methods cylinder (|dz| < 30 um) is the main reading and the code's signed rule
a variant; outcomes are the same under both except per-mouse-table (23 of 24 rows match under the
Methods cylinder, 24 of 24 under the code's rule). Touch bursts are merged by an assumed rule (the
toolbox method is not public). The paper's ± values are 1.4826 times the mean absolute
deviation, not the median absolute deviation the text names. The network model is a
reimplementation of the Methods in numpy and numba, with the background PSP, the sign of
inhibitory PSPs, the stimulus amplitude and the counted population (excitatory neurons) chosen
as stated in its section; its outcomes rest on those choices. Two model outcomes have a change
proposed for a further verification round, as the counted population was revised after the
second round.

What was found about the data set. All 16 archives (24 ablations) are there and readable;
neuron counts and ablated counts equal Extended Data Table 1 in all 24 rows, and the
per-ablation score changes match it within 0.005 in 23 of 24 rows under the Methods cylinder
(several differ by 0.001 to 0.002; all 24 match with the code's cylinder). The
session files come in two MATLAB formats (v5 and v7.3) with no note of it. The data descriptor
names no file other than the session structs and the ablation logs; the curation files that say
which neurons were ablated, and their two layouts (NYU per-attempt flags, JRC id lists), can
only be found through the paper's code, and an250220's curation file lists no attempted neurons.
an278937's pre-ablation directory has a sixth subvolume (995 neurons) with no post-ablation
counterpart, and an278939's post-silent files lack the trial-averaged dF/F. No file in the
archives holds histology. The paper's grand medians for whisking and silent ablations and its behaviour P
values follow its code, which differs from what the text and legends describe.